##<font color=blue>**Problem Statement**

### Business Context

NewsFindr is redefining news discovery by delivering real-time news updates tailored to user interests. Traditional search methods and generic news feeds often lead to information overload and inefficiencies, making it challenging for users to access relevant and trustworthy content efficiently.

To address this, NewsFindr wants to leverage Agentic AI to build an AI-powered news retrieval agent that ensures accuracy and credibility. By utilizing a structured, multi-step approach, the system will provide secure, fair, and explainable recommendations - enhancing user engagement, optimizing content discovery, and improving access to timely and relevant news.

### Objective

- Provide real-time, personalized news retrieval to help users discover relevant
content effortlessly.

- Ensure accuracy and credibility by sourcing news from trusted platforms and minimizing misinformation.

- Improve user engagement through seamless content discovery, reducing information overload.

- Streamline the news consumption process by eliminating outdated and irrelevant content, providing a refined reading experience.

### Data Dictionary

The SQLite database `customer.db` contains a single business table named **`customers`**:

| Column | Type | Description |
| :--- | :--- | :--- |
| `id` | INTEGER (PK, auto-increment) | Internal numeric key for each customer record |
| `customer_id` | TEXT (unique) | External identifier used across systems, e.g. `A88FEC03-C` |
| `name` | TEXT | Customer's first name |
| `email` | TEXT (unique) | Customer's email address - **this is the lookup key used by the whole system** |
| `interests` | TEXT | JSON-style array of topics, e.g. `["Technology", "Automobile"]` |
| `last_updated` | TIMESTAMP | When the record was created / last modified |

> **Important:** the email column is called `email` (not `email_id`), and `interests` is stored as a JSON array string. Both facts matter for the SQL agent prompt and for the deterministic fallback parser later in the notebook.

### Solution Architecture (how the pieces fit together)

The system is a **multi-step agentic pipeline**. Every box below is built and tested in its own section of this notebook, then integrated in the final `query_response()` function.

```
 customer email + user query
            |
            v
 [1] Email guardrail (format check + exists in DB?)          -> stops early for unknown users (0 tokens)
            |
            v
 [2] SQL Agent (LangGraph ReAct + SQLDatabaseToolkit)       -> interests, e.g. ["Technology","Automobile"]
            |      (cross-checked against a direct parameterised SQL query)
            v
 [3] News Agent (LangGraph ReAct) orchestrating 4 tools:
        Tool-1 ExpandSearchQueries  -> time-sensitive search queries (LLM)
        Tool-2 DuckDuckGoSearch     -> live news results (title, url, date, snippet)
        Tool-3 CredibilityFilter    -> blocklist + dedupe + recency sort + LLM credibility ranking
        Tool-4 SummarizeNews        -> top-3 cited summaries (LLM), one per article
            |
            v
 Final personalised briefing with source links   (+ automatic fallback to a fixed pipeline if the agent fails)
```

### How to execute this notebook

1. **Get a Groq API key** (free) from https://console.groq.com/keys.
   - *Colab:* add it under the key icon (Secrets) as `GROQ_API_KEY` and enable notebook access.
   - *Local Jupyter / VS Code / Cursor:* set the environment variable before starting the kernel, e.g. PowerShell `$env:GROQ_API_KEY="gsk_..."`, or simply type it when prompted.
2. **Upload `customer.db`** to `/content/` (Colab) or keep it next to this notebook (local).
3. Run the **install cell** once, **restart the kernel/runtime**, then use *Run All* (or run cells top-to-bottom). Running out of order will raise `NameError`s because later cells depend on earlier objects.
4. Each section ends with a **Verify** cell. If a verify cell prints a warning, fix that step before moving on - later steps depend on it.
5. The full run takes a few minutes: the Groq free tier allows ~8k tokens/minute, so the built-in token budget occasionally pauses execution (you will see `[budget] ... waiting` messages - this is expected, not an error).
6. The **Execution Report** at the end of the notebook is generated from the actual run (timings, tool traces, URLs kept and token usage).

##<font color=blue>**Please read the instructions carefully before starting the project**

This is a commented Python Notebook file in which all the instructions and tasks to be performed are mentioned.
* Blanks '_____' are provided in the notebook that
needs to be filled with an appropriate code to get the correct result. With every '_____' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors.
* Add the results/observations at the end of the analysis and submit the same.

> In this completed version every blank has been filled; each filled line keeps its original `#Complete the code ...` comment so the reviewer can find it quickly.

##<font color=blue>**Installing and Importing Necessary Libraries and Dependencies**

**What this step does:** installs the exact, pinned versions of LangChain, LangGraph, the Groq connector and the DuckDuckGo search client. Pinning versions makes the notebook reproducible - LangChain APIs change frequently between releases.

| Package | Role in NewsFindr |
| :--- | :--- |
| `langchain`, `langchain-core` | Messages, tool abstractions (`StructuredTool`) |
| `langchain-community` | `SQLDatabase` wrapper + `SQLDatabaseToolkit` for the SQL agent |
| `langchain-groq` | `ChatGroq` - the LLM served by Groq |
| `langgraph` | `create_react_agent` - the ReAct agent loop used by both agents |
| `ddgs` | DuckDuckGo search client (news + web) - no API key required |
| `primp==0.15.0` | HTTP client used by `ddgs`. **Pinned on purpose:** `ddgs 9.6.1` requests the `chrome_127` browser profile which newer `primp` releases removed, so without this pin *every* search fails with `primp.BuilderError: Invalid impersonate` |
| `truststore` | Makes Python trust the operating system's certificate store. On corporate networks that inspect HTTPS, every Groq call otherwise fails with `APIConnectionError` (`CERTIFICATE_VERIFY_FAILED`). Harmless on Colab |
| `truststore` | Makes Python trust the operating system's certificate store. Needed on corporate networks that inspect HTTPS traffic - otherwise every Groq call fails with `CERTIFICATE_VERIFY_FAILED` (surfacing as `APIConnectionError: Connection error`). Harmless on Colab |

**How to verify:** the cell finishes without a red `ERROR:` line. Dependency *warnings* can be ignored.

In [1]:
# Single, pinned install cell. Do NOT run any other pip install later in this notebook.
# numpy>=2.1.0  : Python < 3.13 requires NumPy >= 2.1 to resolve dependency issues (harmless on Colab).
# primp==0.15.0 : ddgs 9.6.1 needs the "chrome_127" impersonation profile that newer primp removed.
# (comments are kept on separate lines so the command also works in the Windows shell)
# truststore    : lets Python use the operating-system certificate store (needed behind corporate TLS proxies).
%pip install -qU langchain==0.3.27 langchain-core==0.3.75 langchain-community==0.3.27 langchain-groq==0.3.8 langgraph==0.6.6 ddgs==9.6.1 primp==0.15.0 truststore "numpy>=2.1.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.


**Note:**

- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

###**Imports**

**What this step does:** imports everything used later, grouped by purpose (standard library, data handling, LangChain/LangGraph, validation, web search).

**How to verify:** the cell runs silently. An `ImportError`/`ModuleNotFoundError` means the install cell was not run or the kernel was not restarted afterwards. The version printout confirms the pinned versions are the ones actually loaded.

In [2]:
# Core Python standard utilities for system tasks, parsing, timing, and type annotations
import json
import os
import re
import sys
import time
import random
import sqlite3
from datetime import datetime
from importlib.metadata import version
from typing import List, Dict, Any, Optional

# Data analysis and manipulation library
import pandas as pd
from IPython.display import display, Markdown

# LangChain message abstractions for structuring agent prompts
from langchain_core.messages import SystemMessage, HumanMessage

# Custom tool wrappers for binding functions to LangChain agents
from langchain_core.tools import Tool, StructuredTool
from langchain_groq import ChatGroq

# LangChain database wrappers and automated SQL agent toolkits
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langgraph.prebuilt import create_react_agent

# Pydantic schema validation for strict tool parameter definitions
from pydantic import BaseModel, Field

# DuckDuckGo web search client for real-time web data extraction
from ddgs import DDGS

# Use the OS certificate store for HTTPS (fixes CERTIFICATE_VERIFY_FAILED behind corporate proxies;
# certificate verification stays fully enabled)
try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass

# Suppress non-critical deprecation warnings for cleaner notebook output
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_colwidth", 120)

# Record the environment for the execution report at the end of the notebook
ENV_INFO = {"python": sys.version.split()[0]}
for pkg in ["langchain", "langchain-core", "langchain-community", "langchain-groq",
            "langgraph", "ddgs", "primp"]:
    ENV_INFO[pkg] = version(pkg)
print(ENV_INFO)

{'python': '3.13.16', 'langchain': '0.3.27', 'langchain-core': '0.3.75', 'langchain-community': '0.3.27', 'langchain-groq': '0.3.8', 'langgraph': '0.6.6', 'ddgs': '9.6.1', 'primp': '0.15.0'}


#<font color=blue>**Utilities**

####**Note**

This section from **Token-budget settings** to **Safe LLM call helper** focuses on demonstrating how to apply limits on LLM calls and invocation requests.

- The primary goal here is to show how we can **protect the LLM from being overused or exhausted** by putting sensible controls in place.

- You are *not* required to go through every line of code or deeply understand the full execution flow to benefit from this section. The code is provided mainly as a **safety and performance measure**, not as core learning content.

- Treat this as a reference for good practices in managing LLM usage, rather than a must‑understand code block.

###**Token-budget settings**

**Configuration & Rate Limit Controls**

This block defines central rules for API traffic, text processing limits, and AI execution to keep the application stable, fast, and cost-effective on free-tier services.

| Setting Category | Purpose | Key Variables |
| :--- | :--- | :--- |
| **Rate Limit Protection** | Prevents API blocks by capping tokens sent per minute and auto-retrying requests if rate limits (Error 429) are hit. | `TPM_LIMIT`, `TPM_SAFETY`, `RATE_RETRIES` |
| **Token Budgeting** | Trims search results, body text, and database items so AI prompts stay concise, fast, and within memory caps. | `MAX_INTERESTS`, `MAX_BODY_CHARS`, `LLM_MAX_TOKENS` |
| **Freshness & Output** | Restricts news to the last week and returns the **top 3** articles required by the brief. | `NEWS_TIMELIMIT`, `MAX_URLS_TO_SUMMARIZE` |
| **Workflow Control** | Caps reasoning steps to stop infinite loops and toggles between a flexible AI agent or a lower-cost fixed pipeline. | `USE_AGENT`, `AGENT_RECURSION_LIMIT` |

**Changes vs. the template (and why):**
- `MAX_URLS_TO_SUMMARIZE = 3` - the brief asks for the *top 3 latest* news items per query.
- `MAX_RESULTS_PER_QUERY = 5`, `MAX_BODY_CHARS = 300` - more candidates give the credibility filter enough choice to still return 3 good articles, while shorter snippets keep the total prompt size about the same.
- `NEWS_TIMELIMIT = "w"` - DuckDuckGo News only returns articles from the past week, which removes outdated content (objective 4). If a niche query has no results in that window the search tool widens to the past month automatically.
- `NEWS_REGION = "wt-wt"` + `REGION_BY_KEYWORD` - worldwide results by default, and the India news edition (`in-en`) for queries about India. In testing, the default US region returned US/Canadian politics for an India-focused reader, and even the worldwide region matched "American Indian" articles.
- `AGENT_RECURSION_LIMIT = 16` - one full news-agent run needs ~11 graph steps (expand + 2 searches + filter + summarize + final answer); 12 left almost no headroom.
- `REASONING_EFFORT = "low"` - `gpt-oss` models spend hidden "reasoning" tokens that count against `max_tokens` and the per-minute budget; these tasks are simple, so low effort is enough.

In [3]:
# ---- Groq free-tier rate limits --------------------------------------------
# Base limit for Groq free-tier TPM threshold (adjust per account settings)
TPM_LIMIT      = 8000    # Maximum tokens allowed per minute
TPM_SAFETY     = 0.80    # Safety factor (80% allocation) to account for estimation variance
RATE_RETRIES   = 6       # Maximum backoff attempts when receiving HTTP 429 rate limit errors

# ---- Token budget controls -------------------------------------------------
MAX_INTERESTS          = 2     # Upper bound on user interests queried from SQLite database
MAX_RESULTS_PER_QUERY  = 5     # Maximum search snippet results kept per DuckDuckGo query
MAX_BODY_CHARS         = 300   # Character cutoff per snippet (news snippets are ~200-300 chars anyway)
MAX_RESULTS_TO_FILTER  = 8     # Maximum search results passed to downstream credibility filters
MAX_URLS_TO_SUMMARIZE  = 3     # Top-N latest credible articles summarised per query (brief asks for 3)
AGENT_RECURSION_LIMIT  = 16    # Maximum step count allowed before terminating a ReAct agent loop
LLM_MAX_TOKENS         = 2000  # Token limit capping model response generation length
REASONING_EFFORT       = "low" # gpt-oss hidden reasoning budget ("low" | "medium" | "high")

# ---- Freshness -------------------------------------------------------------
NEWS_TIMELIMIT = "w"           # DuckDuckGo News window: "d" day, "w" week, "m" month
NEWS_REGION    = "wt-wt"       # worldwide results (DuckDuckGo defaults to "us-en", which skews e.g. India news)
REGION_BY_KEYWORD = {r"\bindia": "in-en"}   # queries mentioning India/Indian use the India news edition
TODAY = datetime.now().strftime("%d %B %Y")

# Strategy Switch: True uses autonomous ReAct agent; False uses fixed, low-cost pipeline
USE_AGENT = True
print(f"Config loaded - today is {TODAY}, agent mode = {USE_AGENT}")

Config loaded - today is 14 November 2026, agent mode = True


###**Rate-limit budget**

This script acts like a smart speed regulator for sending messages to the AI. It keeps track of how much text is being sent, prevents sending too much at once, and automatically pauses the program when speed limit warnings occur.

* Usage Tracker: Keeps a running 1-minute log of text sent. If sending a new message will exceed the safety allowance, it pauses the program until the time window resets.
* Length Estimator: Calculates the total word count of incoming and outgoing text before sending it, ensuring requests stay within safe boundaries.
* Error Reader: Detects "slow down" warning messages from the service, reads how many seconds to wait, and pauses safely instead of breaking the app.
* System Reset: Provides a full 60-second pause to completely clear the tracker and start fresh with a full usage allowance.
* Usage Statistics (added): `TOKEN_STATS` counts every LLM call and the real tokens Groq reports, so the final report can show the cost of each test case.

In [4]:
class TokenBudget:
    """Rolling 60-second token budget manager."""

    def __init__(self, tpm_limit: int, safety: float = 0.8):
        self.limit = int(tpm_limit * safety)      # Usable token cap after safety margin
        self.events: List[List[float]] = []        # Tracks execution events as [timestamp, tokens]

    def _prune(self) -> None:
        """Remove usage events older than 60 seconds from the rolling window."""
        cutoff = time.time() - 60
        self.events = [e for e in self.events if e[0] > cutoff]

    def used(self) -> int:
        """Calculate total tokens consumed within the active 60-second window."""
        self._prune()
        return int(sum(e[1] for e in self.events))

    def reserve(self, tokens: int) -> None:
        """Block execution until requested tokens fit into the rolling window."""
        tokens = min(tokens, self.limit)          # Cap reservation request to maximum window limit
        while True:
            self._prune()
            # Proceed if room is available or no prior event history exists
            if self.used() + tokens <= self.limit or not self.events:
                break
            # Calculate sleep duration until the oldest logged event expires
            wait = max(61 - (time.time() - self.events[0][0]), 1.0)
            print(f"  [budget] {self.used()}/{self.limit} tokens used this minute - "
                  f"waiting {wait:.0f}s for the window to refill")
            time.sleep(wait)
        # Record new token allocation event
        self.events.append([time.time(), tokens])

    def settle(self, estimated: int, actual: int) -> None:
        """Reconcile estimated token usage with exact usage returned by API."""
        if self.events and actual > 0:
            self.events[-1][1] = min(actual, self.limit)

    def penalise(self) -> None:
        """Pessimistically mark the budget window as full upon encountering a 429 error."""
        self.events.append([time.time(), self.limit])


# Initialize global rate-limiting budget tracker instance
BUDGET = TokenBudget(TPM_LIMIT, TPM_SAFETY)

# Cumulative usage counters (reported per test case in the execution report)
TOKEN_STATS = {"calls": 0, "tokens": 0}


def estimate_tokens(text: Any) -> int:
    """Estimate token count using a conservative ~3 characters per token ratio."""
    return max(1, len(str(text)) // 3)


def estimate_messages_tokens(messages) -> int:
    """Sum estimated tokens across message contents, formatting overhead, and tool calls."""
    total = 0
    for m in messages:
        content = getattr(m, "content", m)        # Extract content from message object or dict
        total += estimate_tokens(content) + 4      # Include 4-token structural overhead per message
        for tc in (getattr(m, "tool_calls", None) or []):
            total += estimate_tokens(tc)          # Add token costs for attached tool call payloads
    return total


def parse_retry_after(error_text: str, default: float = 20.0) -> float:
    """Extract recommended backoff duration in seconds from API rate limit error strings."""
    # Pattern match combined minutes and seconds format first (e.g., "1m30.5s")
    m = re.search(r"try again in (\d+)m([\d.]+)s", error_text, re.I)
    if m:
        return float(m.group(1)) * 60 + float(m.group(2)) + 2.0

    # Pattern match standard seconds format (e.g., "3.53s" or "500ms")
    m = re.search(r"try again in ([\d.]+)\s*m?s", error_text, re.I)
    if m:
        secs = float(m.group(1))
        if "ms" in error_text[m.start():m.end() + 3].lower():
            secs /= 1000.0                        # Convert milliseconds to seconds
        return secs + 2.0                         # Add safety buffer to recommended wait

    return default                                # Return fallback wait duration if parsing fails


def is_rate_limit(err: Exception) -> bool:
    """Check if exception message matches common rate-limit or 429 indicator strings."""
    msg = str(err).lower()
    return any(k in msg for k in ["rate limit", "rate_limit", "429", "too many requests"])


def cooldown(seconds: int = 60) -> None:
    """Pause execution and reset event history to clear the rolling token allowance."""
    print(f"Cooling down for {seconds}s to reset the per-minute allowance...")
    time.sleep(seconds)
    BUDGET.events.clear()                        # Clear stored usage events
    print("Ready.")


# Output current usable token budget limit
print(f"Token budget: {BUDGET.limit} tokens/min (of {TPM_LIMIT} allowed)")

Token budget: 6400 tokens/min (of 8000 allowed)


###**Calling Groq API Key**

This step securely fetches your API key so the AI service can be accessed, trying a few different ways depending on where the notebook is running.

* First Try: Checks if the key is already saved as an environment variable.
* Second Try: If not found, checks Google Colab's secure secrets storage.
* Last Resort: If neither works, asks you to type the key in directly (hidden as you type, like a password).
* Safety Check: Stops the notebook with a clear error if no key was ever provided, so you don't run into confusing failures later on.

**Security note:** the key is never printed or hard-coded in the notebook, so the exported HTML can be shared safely.

In [5]:
# Primary check: search for existing environment variable
groq_api_key = os.environ.get("GROQ_API_KEY")

if not groq_api_key:
    try:
        # Secondary check: attempt retrieval from Google Colab Secrets
        from google.colab import userdata
        groq_api_key = userdata.get("GROQ_API_KEY")
    except Exception:
        # Fallback check: prompt user for masked interactive password input
        from getpass import getpass
        groq_api_key = getpass("Enter your GROQ_API_KEY: ")

# Export API key into environment variables for global access across packages
os.environ["GROQ_API_KEY"] = groq_api_key or ""

# Enforce non-empty key validation to halt execution if credentials are missing
assert os.environ["GROQ_API_KEY"], "GROQ_API_KEY is not set."
print("Groq API key loaded.")

Groq API key loaded.


### **Rate-Limited LLM Wrapper & Initialization**

This step creates the AI connection itself, wrapped with the speed-limiter logic built earlier, and prepares two versions of it for different needs.

* Model Choice: Sets which AI model to use, with a lighter backup option noted in case the first one has stricter usage limits.
* Speed-Aware AI: Builds a version of the AI connection that automatically checks the usage tracker before every request, waits if needed, and retries safely if a "slow down" response comes back.
* Two Personalities:
  - **Precise Mode** (`llm`, temperature 0): Used for tasks like database lookups, filtering and summarising, where answers need to be consistent and predictable.
  - **Creative Mode** (`llm_high`, temperature 0.7): Used when more varied or natural-sounding phrasing is wanted.
* Ready Check: Confirms both AI connections are set up and ready to use.

**Why subclass `ChatGroq`?** Agents built by LangGraph call the model many times internally. Putting the rate limiting inside `_generate` means *every* call - including the hidden ones the agents make - goes through the budget, not only the calls we write ourselves.

In [6]:
# Default model selection for pipeline and agent execution
MODEL_NAME = "openai/gpt-oss-120b"
# Alternative fallback model for lower token ceilings:
# MODEL_NAME = "openai/gpt-oss-20b"


class RateLimitedChatGroq(ChatGroq):
    """ChatGroq wrapper that embeds rate-limiting and retry logic inside the LLM generation step."""

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        # Calculate pessimistic pre-call estimate (input tokens + response allocation cap)
        estimated = estimate_messages_tokens(messages) + (self.max_tokens or 512)

        for attempt in range(RATE_RETRIES):
            # Reserve token allocation before making the API request
            BUDGET.reserve(estimated)
            try:
                result = super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)

                # Reconcile estimated cost against actual usage returned by API response
                try:
                    usage = (result.llm_output or {}).get("token_usage", {})
                    actual = usage.get("total_tokens", 0)
                    TOKEN_STATS["calls"] += 1
                    if actual:
                        BUDGET.settle(estimated, int(actual))
                        TOKEN_STATS["tokens"] += int(actual)
                except Exception:
                    pass  # Non-critical metric collection failure

                return result

            except Exception as e:
                # Re-raise error if not a 429 rate limit or if retry attempts are exhausted
                if not is_rate_limit(e) or attempt == RATE_RETRIES - 1:
                    raise

                # Penalize rate limiter and wait out backoff period requested by server
                BUDGET.penalise()
                wait = parse_retry_after(str(e))
                print(f"  [429] rate limited - waiting {wait:.1f}s "
                      f"(attempt {attempt + 1}/{RATE_RETRIES})")
                time.sleep(wait)

        raise RuntimeError("Exhausted rate-limit retries")


# Initialize deterministic LLM instance for standard pipeline and database tasks
llm = RateLimitedChatGroq(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=LLM_MAX_TOKENS,
    reasoning_effort=REASONING_EFFORT,
    max_retries=0,                  # Disable native retries to let RateLimitedChatGroq handle backoffs
    groq_api_key=groq_api_key,
)

# Initialize creative LLM instance for varied text generation
llm_high = RateLimitedChatGroq(
    model=MODEL_NAME,
    temperature=0.7,                # Higher temperature for diverse phrasing
    max_tokens=LLM_MAX_TOKENS,
    reasoning_effort=REASONING_EFFORT,
    max_retries=0,
    groq_api_key=groq_api_key,
)

print("Rate-limited LLMs ready:", MODEL_NAME)

Rate-limited LLMs ready: openai/gpt-oss-120b


###**Safe LLM call helper**

This script makes sure a message actually gets through, even if the connection hiccups along the way.

* Retry Helper: If sending a message fails because of a temporary issue (like a timeout or the service being briefly overloaded), it waits a bit and tries again - up to a set number of attempts.
* Smart Waiting: Each retry waits a little longer than the last, so it doesn't hammer the service repeatedly in a short burst.
* Give-Up Point: If it's not a temporary issue, or all attempts are used up, it stops trying and reports the problem instead of getting stuck.
* Text Cleaner: Tidies up extra spaces and trims text down to a safe length, so what gets sent stays within limits.

In [7]:
def safe_llm_call(messages, model=None, retries: int = 3) -> str:
    """Invoke the LLM and return its text.

    Rate limiting is handled inside RateLimitedChatGroq._generate, so this only needs to
    cover non-rate-limit transients (timeouts, 502/503, dropped connections).
    """
    model = model or llm          # default to the module-level deterministic LLM
    last_err = None
    for attempt in range(retries):
        try:
            return model.invoke(messages).content
        except Exception as e:
            last_err = e
            msg = str(e).lower()
            # note: "rate limit" is included here as a safety net, though _generate
            # should already have handled it before the exception reaches this point
            transient = any(k in msg for k in
                            ["timeout", "overloaded", "503", "502", "connection", "rate limit"])
            if not transient or attempt == retries - 1:
                break          # non-transient error, or out of retries - stop and report
            wait = 5.0 * (2 ** attempt) + random.uniform(0, 2)   # exponential backoff + jitter
            print(f"  [retry {attempt + 1}/{retries}] {type(e).__name__} - waiting {wait:.1f}s")
            time.sleep(wait)
    print(f"  [LLM error] {last_err}")
    return ""          # fail soft - caller gets an empty string instead of a crash


def clip(text: str, n: int) -> str:
    """Collapse whitespace and hard-truncate. Keeps prompts small and predictable."""
    text = re.sub(r"\s+", " ", str(text or "")).strip()   # normalize None/whitespace-only input too
    return text[:n]        # hard cap on length, not on token count - keep prompts bounded

###**Verify: LLM responds to a simple query**

**What this step does:** sends one tiny prompt through the full stack (`safe_llm_call` -> `RateLimitedChatGroq` -> Groq API) before we build anything on top of it.

**How to verify:** you should see a one-sentence answer and a latency of a few seconds. An authentication error means the API key is wrong; an empty answer means the model name is unavailable on your account (switch to `openai/gpt-oss-20b`).

In [8]:
t0 = time.time()
reply = safe_llm_call([
    SystemMessage(content="You are a concise assistant. Answer in one sentence."),
    HumanMessage(content="What is an AI agent, and how is it different from a plain chatbot?"),
])
print("LLM reply :", reply)
print(f"Latency   : {time.time() - t0:.1f}s | calls so far: {TOKEN_STATS['calls']} | tokens so far: {TOKEN_STATS['tokens']}")
assert reply.strip(), "LLM returned an empty response - check the API key / model name."

LLM reply : An AI agent is an autonomous system that perceives its environment, makes decisions, and takes actions to achieve goals—often integrating planning, learning, and tool use—whereas a plain chatbot simply processes user inputs to generate conversational replies without independent goal‑directed behavior.
Latency   : 0.4s | calls so far: 1 | tokens so far: 170


#<font color=blue>**Building an SQL agent**

###Customer database

This step connects to the customer database and checks that it's actually loaded correctly before moving on.

* Locate the Database: Automatically points to the right file location, whether running in Google Colab or locally.
* Connect: Links the database to LangChain so the AI can query it. SQLAlchemy expects a URI of the form `sqlite:///<path>` (three slashes = relative or absolute file path).
* Check Tables: Lists out what tables are available inside the database.
* Safety Check: Stops the notebook right away with a clear error if the database turns out to be empty or missing, instead of letting confusing errors show up later.

In [9]:
import os
from google.colab import drive
drive.mount('/content/drive')

print("Current directory:", os.getcwd())
print("Files:", os.listdir("."))

# Search for customer.db
#for root, dirs, files in os.walk("/content"):
#    if "customer.db" in files:
#        print("Found:", os.path.join(root, "customer.db"))

DB_PATH = "/content/drive/MyDrive/Colab Notebooks/News-Paper-Proj-LangChain-Agentic/customer.db" if os.path.isdir("/content") else "customer.db"   # Colab vs local path
assert os.path.exists(DB_PATH), f"{DB_PATH} not found - upload customer.db first."

db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")                                   #Complete the code by providing the SQLite connection string built from DB_PATH

TABLES = db.get_usable_table_names()   # tables the LLM is allowed to see/query
print("Tables:", TABLES)
assert TABLES, "Database has no tables - check DB_PATH."   # fail fast if the DB is empty/missing

# Column names used throughout the notebook (they must match the real schema)
TABLE_NAME = "customers"
EMAIL_COL = "email"          # the column is `email` - not `email_id`
INTEREST_COL = "interests"

Mounted at /content/drive
Current directory: /content
Files: ['.config', 'drive', 'sample_data']
Tables: ['customers']


###Explore the data (ground truth)

**What this step does:** reads the table directly with pandas (no LLM) so we know what the *correct* answers are. Every later LLM output (emails, interests) is checked against this ground truth.

**How to verify:** 15 customers, each with 2-3 interests stored as a JSON array. The interest frequency table shows which topics dominate - useful for choosing diverse test users.

In [10]:
print(db.get_table_info([TABLE_NAME]))      # the schema + sample rows the SQL agent will "see"

with sqlite3.connect(DB_PATH) as conn:
    customers_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)

customers_df["interests_list"] = customers_df[INTEREST_COL].apply(json.loads)   # JSON text -> Python list
print(f"\nRows: {len(customers_df)} | unique emails: {customers_df[EMAIL_COL].nunique()}")
display(customers_df[["id", "customer_id", "name", EMAIL_COL, "interests_list", "last_updated"]])

print("Interest frequency across customers:")
display(customers_df["interests_list"].explode().value_counts().to_frame("customers"))


CREATE TABLE customers (
	id INTEGER, 
	customer_id TEXT NOT NULL, 
	name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	interests TEXT NOT NULL, 
	last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP, 
	PRIMARY KEY (id), 
	UNIQUE (customer_id), 
	UNIQUE (email)
)

/*
3 rows from customers table:
id	customer_id	name	email	interests	last_updated
1	F8641860-7	Kevin	kevin.f8641860-7@gmail.com	["Politics", "Startups", "Travel"]	2025-03-26 06:46:15
2	203631A0-B	Ian	ian.203631a0-b@gmail.com	["Startups", "Travel"]	2025-03-26 06:46:15
3	D77D96F3-3	Julia	julia.d77d96f3-3@gmail.com	["India", "Automobile", "Business"]	2025-03-26 06:46:15
*/

Rows: 15 | unique emails: 15


,id,customer_id,name,email,interests_list,last_updated
0,1,F8641860-7,Kevin,kevin.f8641860-7@gmail.com,"[Politics, Startups, Travel]",2025-03-26 06:46:15
1,2,203631A0-B,Ian,ian.203631a0-b@gmail.com,"[Startups, Travel]",2025-03-26 06:46:15
2,3,D77D96F3-3,Julia,julia.d77d96f3-3@gmail.com,"[India, Automobile, Business]",2025-03-26 06:46:15
3,4,6EB33C45-5,Alice,alice.6eb33c45-5@gmail.com,"[Politics, Technology, Business]",2025-03-26 06:46:15
4,5,EDD38E10-6,Oscar,oscar.edd38e10-6@gmail.com,"[Automobile, India, Sports]",2025-03-26 06:46:15
5,6,4770B814-2,Hannah,hannah.4770b814-2@gmail.com,"[Entertainment, Business, Technology]",2025-03-26 06:46:15
6,7,5483CB53-8,George,george.5483cb53-8@gmail.com,"[India, Politics, Science]",2025-03-26 06:46:15
7,8,FA8FE05B-7,Lily,lily.fa8fe05b-7@gmail.com,"[India, Science]",2025-03-26 06:46:15
8,9,C63FA237-E,Nora,nora.c63fa237-e@gmail.com,"[Entertainment, Business]",2025-03-26 06:46:15
9,10,45E01919-0,Lily,lily.45e01919-0@gmail.com,"[India, India, Politics]",2025-03-26 06:46:15


Interest frequency across customers:


,customers
interests_list,
India,7
Entertainment,5
Politics,4
Technology,4
Automobile,4
Business,4
Startups,3
Travel,2
Sports,2


This step gives the AI a toolkit for working with the customer database.

* Toolkit Setup: Connects the database and the AI together, creating a set of ready-made tools the AI can use.
* Available Tools: These tools let the AI look at the database structure, check if a query is written correctly, and run queries to get results.
* Careful Naming: Saved as its own separate name so it doesn't accidentally get overwritten later when other tools are created.
* Verification: Prints out the names of these tools, just to confirm everything is set up and available for use.

Expected tools: `sql_db_query`, `sql_db_schema`, `sql_db_list_tables`, `sql_db_query_checker`.

In [11]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)   # gives the LLM query/schema-inspection tools for `db`
sql_tools = toolkit.get_tools()          # named sql_tools (not tools) so it doesn't collide
                                         # notebook reused `tools` later and clobbered these
for t in sql_tools:
    print(f"{t.name:22s} -> {t.description[:90]}...")   # sanity check - confirm the expected SQL tools were created

sql_db_query           -> Input to this tool is a detailed and correct SQL query, output is a result from the databa...
sql_db_schema          -> Input to this tool is a comma-separated list of tables, output is the schema and sample ro...
sql_db_list_tables     -> Input is an empty string, output is a comma-separated list of tables in the database....
sql_db_query_checker   -> Use this tool to double check if your query is correct before executing it. Always use thi...


###SQL Agent

This step gives the model clear instructions and turns it into a working database assistant.

* Instructions: Tells the model exactly how to behave - fetch the requested data and reply with just the values, no extra explanation.
* Assistant Builder: Combines the AI, the database tools, and these instructions into one ready-to-use assistant, scoped only to the database tools.
* Compatibility Handling: Quietly adjusts for small differences between versions of the underlying library, so this works regardless of which version is installed.
* Ready Check: Confirms the database assistant has been created successfully.

**Design of the system message:**
| Rule | Why |
| :--- | :--- |
| Schema is given up-front | Saves the agent 1-2 tool calls (and tokens) spent discovering the table |
| Read-only (`SELECT` only) | Security - an LLM with database access must never be able to modify or delete customer data |
| Exact email match, `NOT_FOUND` sentinel | Prevents the agent from "guessing" a similar customer (fairness / privacy) and gives code an easy string to check |
| Values only, no prose | Downstream code parses the answer, so commentary would break parsing |

In [12]:
sql_system_message = """You are a read-only SQL assistant for the NewsFindr SQLite database.
The database has ONE table, `customers`, with columns:
  id (INTEGER), customer_id (TEXT), name (TEXT), email (TEXT, unique),
  interests (TEXT holding a JSON array such as ["Technology", "Automobile"]), last_updated (TIMESTAMP).

Rules:
1. The schema above is complete - write the query directly and run it with sql_db_query.
2. Only run SELECT statements. Never run INSERT, UPDATE, DELETE, DROP, ALTER, CREATE or PRAGMA.
3. When filtering by email use an exact match: WHERE LOWER(email) = LOWER('<email>'). Never guess or
   return a different customer.
4. If the query returns no rows, answer exactly: NOT_FOUND
5. Reply with only the requested values - no explanations, no SQL, no markdown.
   Several values -> comma-separated. If the user asks for JSON, return valid JSON only.
"""                                                                                  #Complete the code by writing a system message that tells the SQL assistant how to respond

def build_agent(model, tool_list, system_message):
    """create_react_agent renamed this argument between LangGraph versions
    (state_modifier -> prompt), so try both instead of pinning to one."""
    try:
        return create_react_agent(model, tool_list, prompt=system_message)   # newer LangGraph API
    except TypeError:
        return create_react_agent(model, tool_list, state_modifier=system_message)   # older LangGraph API

db_agent = build_agent(llm, sql_tools, sql_system_message)                          #Complete the code by calling the llm
print("SQL agent ready.")

SQL agent ready.


Now that the SQLite database connection is established and the SQL toolset is loaded, this step wraps those tools into a dedicated, version-compatible ReAct agent (db_agent). By constraining its system prompt to output clean, unformatted text without commentary, the agent is prepared to cleanly retrieve user profiles and preferences required by downstream search and recommendation steps.

###Database Verification

With the db_agent fully constructed, this step runs a test query against the database to extract all unique user email addresses while enforcing recursion limits (AGENT_RECURSION_LIMIT) to prevent loop runaway. Verifying clean database retrieval here ensures the pipeline can reliably lookup user profiles, unlocking customer preference extraction in the next stage.

**How to verify:** the second part of the cell compares the agent's answer with the pandas ground truth. Expect `15/15 emails match`. We also print the SQL the agent actually executed - this is the *explainability* of the agent: you can see exactly what it did.

In [13]:
# Define test query targeting unique user identifier records
query = "Fetch all the unique email values"

# Invoke SQL agent using LangGraph dictionary schema while enforcing recursion limits
response = db_agent.invoke(
    {"messages": [HumanMessage(content=query)]},
    config={"recursion_limit": AGENT_RECURSION_LIMIT},   # Cap ReAct loop steps to avoid runaway API calls
)

# Extract and print the final message content containing the comma-separated output
agent_answer = response["messages"][-1].content
print(agent_answer)

# ---- Verify against ground truth + show the SQL the agent ran -------------------
for m in response["messages"]:
    for tc in (getattr(m, "tool_calls", None) or []):
        print(f"\n[agent tool call] {tc['name']}: {tc['args']}")

agent_emails = {e.lower() for e in re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", agent_answer)}
db_emails = set(customers_df[EMAIL_COL].str.lower())
print(f"\n{len(agent_emails & db_emails)}/{len(db_emails)} emails match the database; "
      f"unexpected: {sorted(agent_emails - db_emails) or 'none'}")

alice.6eb33c45-5@gmail.com, alice.d56e53b6-1@gmail.com, emma.a88fec03-c@gmail.com, george.5483cb53-8@gmail.com, hannah.4770b814-2@gmail.com, ian.203631a0-b@gmail.com, ian.aee37571-7@gmail.com, julia.b5a14512-c@gmail.com, julia.cf4d1edb-9@gmail.com, julia.d77d96f3-3@gmail.com, kevin.f8641860-7@gmail.com, lily.45e01919-0@gmail.com, lily.fa8fe05b-7@gmail.com, nora.c63fa237-e@gmail.com, oscar.edd38e10-6@gmail.com

[agent tool call] sql_db_query: {'query': 'SELECT DISTINCT email FROM customers'}

15/15 emails match the database; unexpected: none


###Verify the customer's email and retrieve their details

The rubric asks the SQL agent to *verify the customer's email* before retrieving details. We use a layered approach:

1. **Format check (regex)** - rejects malformed input instantly. Because only `[A-Za-z0-9._%+-]` characters can pass, this also blocks **prompt / SQL injection** attempts such as `x' OR 1=1 --` before the text ever reaches the LLM.
2. **Existence check (parameterised SQL)** - a deterministic `SELECT 1 ... WHERE email = ?` costs zero tokens and cannot be fooled by the LLM.
3. **Retrieval via the SQL agent** - only for verified customers, the agent fetches `customer_id`, `name`, `email`, `interests` as JSON.

**How to verify:** the valid email returns `verified: True` with details that match the table above; the unknown email returns `email not found`; the malformed one returns `invalid email format` - and the last two consume no LLM tokens.

In [14]:
EMAIL_REGEX = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")


def email_exists(email: str) -> bool:
    """Deterministic, parameterised existence check (no LLM, injection-safe)."""
    with sqlite3.connect(DB_PATH) as conn:
        row = conn.execute(
            f"SELECT 1 FROM {TABLE_NAME} WHERE LOWER({EMAIL_COL}) = LOWER(?) LIMIT 1", (email,)
        ).fetchone()
    return row is not None


def get_customer_details(email: str) -> Dict[str, Any]:
    """Verify the email, then let the SQL agent retrieve the customer's profile."""
    email = (email or "").strip()
    if not EMAIL_REGEX.match(email):
        return {"email": email, "verified": False, "reason": "invalid email format"}
    if not email_exists(email):
        return {"email": email, "verified": False, "reason": "email not found"}

    result = db_agent.invoke(
        {"messages": [HumanMessage(content=(
            f"Return customer_id, name, email and interests for the customer whose email is "
            f"'{email}'. Respond with one JSON object using exactly those keys; "
            f"interests must be a JSON list."
        ))]},
        config={"recursion_limit": AGENT_RECURSION_LIMIT},
    )
    text = result["messages"][-1].content
    match = re.search(r"\{.*\}", text, re.S)
    try:
        details = json.loads(match.group(0)) if match else {"raw": text}
    except json.JSONDecodeError:
        details = {"raw": text}
    if isinstance(details.get("interests"), str):           # agent may return the raw JSON text
        try:
            details["interests"] = json.loads(details["interests"])
        except json.JSONDecodeError:
            pass
    return {"email": email, "verified": True, "details": details}


for test_email in ["emma.a88fec03-c@gmail.com",      # valid customer
                   "not.a.customer@example.com",     # well-formed but unknown
                   "x' OR 1=1 --"]:                  # malformed / injection attempt
    print(json.dumps(get_customer_details(test_email), indent=2))

{
  "email": "emma.a88fec03-c@gmail.com",
  "verified": true,
  "details": {
    "customer_id": "A88FEC03-C",
    "name": "Emma",
    "email": "emma.a88fec03-c@gmail.com",
    "interests": [
      "Technology",
      "Automobile"
    ]
  }
}
{
  "email": "not.a.customer@example.com",
  "verified": false,
  "reason": "email not found"
}
{
  "email": "x' OR 1=1 --",
  "verified": false,
  "reason": "invalid email format"
}


#<font color=blue>**Define Tools**

The news side of the system is made of **four tools**. Each is a plain Python function wrapped in a `StructuredTool` with a Pydantic input schema, so that:

- we can **unit-test each tool on its own** (smoke tests below), and
- the agent receives a precise description + argument schema for every tool, which reduces malformed tool calls.

A shared `CURRENT_RUN` dictionary records what each tool did during a request (queries, raw hits, kept URLs, summary). It is used for **explainability** (the report at the end shows the full trace) and as a safety net if the agent's final message is incomplete.

In [15]:
# Per-request trace shared by the tools (reset at the start of every query_response call)
CURRENT_RUN: Dict[str, Any] = {}
RUN_LOG: List[Dict[str, Any]] = []      # one entry per end-to-end request, used by the final report

##<font color=blue>**Tool-1: Expand search queries**

- Now that user information can be retrieved from the database, the pipeline needs to convert raw customer interests into focused search keywords.
- This step defines a function that calls the LLM once to transform interest keywords into targeted, time-sensitive news search queries, complete with fallback logic to prevent execution stalls.
- Packaging this function into a StructuredTool allows agents in the upcoming search stage to dynamically generate queries.

**Prompt design:** one query per interest (so every interest is covered - fairness across topics), each combining the interest with the user's focus, and anchored in time with today's month/year so the search engine favours fresh articles. The output must be a bare JSON array so it can be parsed reliably; if parsing fails, a deterministic `"<interest> latest news"` fallback is used.

In [16]:
# Set up input formatting rules for the tool
class ExpandSearchQueriesInput(BaseModel):
    interests: List[str] = Field(description="List of user interests to expand.")
    user_query: str = Field(description="The specific user query or topic to consider.")


def expand_search_queries(interests: List[str], user_query: str) -> str:
    """Expand user interests into time-sensitive news search queries using a single LLM call."""
    # Normalize comma-separated string inputs into list format if necessary
    if isinstance(interests, str):
        interests = [i.strip() for i in interests.split(",") if i.strip()]

    # Apply character and count limits to constrain prompt token usage
    interests = [clip(i, 60) for i in interests][:MAX_INTERESTS]
    if not interests:
        return json.dumps([])

    # System prompt directing the LLM to output a clean JSON array of search strings
    system_prompt = f"""You convert a reader's interests into web NEWS search queries.
Today is {TODAY}.
Return ONLY a JSON array of strings - exactly one query per interest, in the same order. No prose.
Each query must:
- combine the interest with the reader's focus so results are specific, not generic;
- be time-sensitive: include "latest" and the current month and year;
- be 4 to 10 words, plain keywords only (no quotes, site: or boolean operators).
Example output: ["latest electric SUV launches India November 2026", "latest AI chip announcements November 2026"]"""   #Complete the code by providing clear instructions for generating time-sensitive search queries
    prompt = f"Interests: {json.dumps(interests)}\nUser focus: {user_query}"

    # Invoke LLM using the safe execution wrapper
    raw = safe_llm_call([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    # Extract JSON array from LLM response text using regex matching
    queries = []
    match = re.search(r"\[.*\]", raw, re.S)
    if match:
        try:
            parsed = json.loads(match.group(0))
            queries = [str(q).strip() for q in parsed if str(q).strip()]
        except json.JSONDecodeError:
            pass   # Fall through to fallback list on JSON parsing error

    # Fallback query generation to ensure downstream execution never stalls
    if not queries:
        queries = [f"{i} latest news" for i in interests]

    queries = queries[:MAX_INTERESTS]
    CURRENT_RUN["queries"] = queries                 # trace for the report
    # Return final JSON string capped at MAX_INTERESTS limit
    return json.dumps(queries)


# Register function as a structured LangChain tool with Pydantic validation
expand_tool = StructuredTool.from_function(
    func=expand_search_queries,
    name="ExpandSearchQueries",
    description=("Expands user interests into time-sensitive news search queries. "
                 "Input: interests (list of strings) and user_query (string). "
                 "Returns a JSON array of query strings."),
    args_schema=ExpandSearchQueriesInput,   # Enforce input validation schema
)
print("expand_tool ready")

expand_tool ready


Having built the expand_search_queries tool and defined its input formatting rules, this step runs a quick verification test (smoke test) to confirm the function returns a valid JSON array of expanded search terms. Verifying this logic directly ensures search queries are properly formatted before feeding them into the live web search stage.

**How to verify:** the output is a JSON list with one query per interest, each containing "latest" and the current month/year.

In [17]:
smoke_queries = json.loads(expand_search_queries(["Artificial Intelligence", "Robotics"], "latest innovations"))
print(smoke_queries)
assert isinstance(smoke_queries, list) and len(smoke_queries) == 2, "expected one query per interest"

['latest AI innovations November 2026', 'latest robotics innovations November 2026']


##<font color=blue>**Tool-2: Fetch News Results Using DuckDuckGo**

Building on the expanded search queries generated in the previous step, this stage executes live web searches to gather real-time news articles before passing them to the credibility filter.

- Live Web Search Execution: Queries DuckDuckGo News using expanded search terms to fetch dated, relevant news articles.
- Freshness (added): `timelimit=NEWS_TIMELIMIT` restricts results to the past week; if nothing is found the window widens to a month, and only then to general web search.
- Recency ordering (added): results are sorted newest-first by their publication date.
- Token Budget Control: Truncates snippets and formats results as clean JSON strings to satisfy LangGraph tool nodes while keeping token usage low.
- Data Normalization: Cleans snippet text, formats publication dates, and standardizes key names across API responses.
- Source Caching: Stores retrieved hits in a global dictionary (SEARCH_CACHE) to prevent downstream AI agents from hallucinating fake URLs.
- Structured Tool Packaging: Wraps the search logic into ddg_search_tool with explicit parameter validation to ensure smooth agent tool calls.

**How to verify:** the smoke test prints a JSON array of real articles with recent `date` values. An empty `[]` usually means DuckDuckGo is temporarily throttling - wait a minute and re-run.

In [18]:
# Set up input formatting rules for the search tool
class SearchInput(BaseModel):
    query: str = Field(description="A single news search query string.")


def _normalise(r: dict) -> Optional[Dict[str, str]]:
    """Clean raw search results and standardize dictionary key names."""
    url = r.get("href") or r.get("url") or r.get("link") or ""   # Extract URL across different key structures
    if not url:
        return None                                              # Skip results without a valid URL

    title = clip(r.get("title", ""), 100)
    body = clip(r.get("body") or r.get("excerpt") or "", MAX_BODY_CHARS)
    date = str(r.get("date", ""))[:10]                            # Format date as YYYY-MM-DD
    if not re.match(r"\d{4}-\d{2}-\d{2}$", date):
        date = ""                                                 # drop relative dates ("6 days ago") so sorting stays valid
    source = clip(r.get("source", ""), 40)                        # Publisher name (news endpoint only)
    return {"title": title, "url": url, "body": body, "date": date, "source": source}


def ddg_search(query: str) -> str:
    """Search DuckDuckGo News and return top results as a JSON string."""
    query = clip(query, 200)
    results: List[Dict[str, str]] = []
    # Country-specific interests get that country's news edition; everything else is worldwide
    region = next((r for k, r in REGION_BY_KEYWORD.items() if re.search(k, query, re.I)), NEWS_REGION)

    try:
        with DDGS() as ddgs:
            raw = []
            # Primary attempt: news endpoint, newest window first (week -> month)
            for window in dict.fromkeys([NEWS_TIMELIMIT, "m"]):
                try:
                    raw = list(ddgs.news(query, max_results=MAX_RESULTS_PER_QUERY,     #Complete the code by providing the max_results value (hint: use MAX_RESULTS_PER_QUERY)
                                         timelimit=window, region=region))
                except Exception:
                    raw = []
                if raw:
                    break
            if not raw:
                # Fallback attempt: general web search if news search returns no hits
                raw = list(ddgs.text(query, max_results=MAX_RESULTS_PER_QUERY))
            for r in raw:
                item = _normalise(r)
                if item:
                    results.append(item)
    except Exception as e:
        print(f"  [search failed] {query[:50]!r}: {type(e).__name__}")
        return json.dumps([])                 # Return empty list on failure to avoid pipeline stalls

    results.sort(key=lambda r: r["date"], reverse=True)          # newest first (ISO dates sort lexically)

    # Store retrieved results in global cache keyed by URL to verify source authenticity
    for r in results:
        SEARCH_CACHE[r["url"]] = r
    CURRENT_RUN["raw_results"] = CURRENT_RUN.get("raw_results", 0) + len(results)

    return json.dumps(results, ensure_ascii=False)


# Global store for caching verified search hits
SEARCH_CACHE: Dict[str, Dict[str, str]] = {}

# Register search function as a LangChain structured tool with validated schema
ddg_search_tool = StructuredTool.from_function(
    func=ddg_search,
    name="DuckDuckGoSearch",
    description=("Searches DuckDuckGo News for recent articles. Input: query (a single "
                 f"string). Returns a JSON array of up to {MAX_RESULTS_PER_QUERY} results "
                 "with title, url, date and a short body."),
    args_schema=SearchInput,
)

# Run a smoke test and show the results as a table
smoke_hits = json.loads(ddg_search("artificial intelligence breakthrough"))
display(pd.DataFrame(smoke_hits)[["date", "source", "title", "url"]])

,date,source,title,url
0,2026-11-12,Yahoo Finance,Global Legal Technology Innovators Honored in 7th Annual LegalTech Breakthrough Awards Program,https://finance.yahoo.com/technology/ai/articles/global-legal-technology-innovators-honored-130000743.html
1,2026-11-11,India TV News,"AI to bring medical breakthrough? Anthropic CEO claims cancer, Alzheimer's will be history in 10 yea",https://www.indiatvnews.com/news/world/ai-to-bring-medical-breakthrough-anthropic-ceo-promises-cancer-alzheimer-s-wi...
2,2026-11-10,Tom's Guide,AI is changing cancer treatment — and a new breakthrough shows just how far we've come,https://www.msn.com/en-us/technology/artificial-intelligence/ai-is-changing-cancer-treatment-and-a-new-breakthrough-...
3,2026-11-09,note,What Happened When We Gave AI 'Sleep'? ~Learning from the Baby Brain: A Breakthrough in Next-Generat,https://note.com/witty_poppy6556/n/n1d016b89760f?hl=en
4,2026-10-30,Decrypt,Artificial Intelligence,https://decrypt.co/news/artificial-intelligence


##<font color=blue>**Tool-3: Filter Relevant and Trustworthy URLs Based on User Interests**

Following the live DuckDuckGo web search step, this stage processes raw search outputs to eliminate duplicate domains, social media spam, and low-credibility sources. By asking the LLM to return only array index numbers rather than full text, it filters candidate links efficiently before passing clean news sources to the summarization pipeline.

The filter works in **two layers** - cheap deterministic rules first, LLM judgement second:

| Layer | Check | Purpose |
| :--- | :--- | :--- |
| Deterministic | URL must exist in `SEARCH_CACHE` | Drops any URL the agent invented (anti-hallucination) |
| Deterministic | Domain blocklist (social media, video, forums, Q&A sites, search-redirect links, podcast pages) | Removes user-generated / unverified / non-article content |
| Deterministic | One result per domain, newest first | Diversity of sources + recency |
| LLM | Credibility + relevance to the reader's interests, ranked | Editorial judgement that rules can't express |

The agent only needs to pass the **URLs** to this tool; titles and snippets are restored from the cache. This keeps the agent's tool-call arguments short (fewer output tokens, no risk of the model mangling a long JSON payload).

In [19]:
def _coerce_results(search_results: Any) -> List[Dict[str, Any]]:
    """Convert raw input payloads (strings, dicts, lists) into a standardized list of dictionary records."""
    if isinstance(search_results, list):
        out = []
        for r in search_results:
            if isinstance(r, dict):
                out.append(r)
            elif isinstance(r, str):
                out.extend(_coerce_results(r))       # Recurse if list items are JSON strings / URLs
        return out

    if isinstance(search_results, dict):
        # Unwrap nested payload keys commonly returned by agents
        for key in ("results", "items", "sources", "data", "search_results", "urls"):
            if key in search_results and isinstance(search_results[key], (list, str)):
                return _coerce_results(search_results[key])
        return [search_results] if search_results.get("url") else []

    if isinstance(search_results, str):
        items: List[Dict[str, Any]] = []
        for blob in re.findall(r"\[.*?\]", search_results, re.S):      # Extract JSON array substrings
            try:
                parsed = json.loads(blob)
                items.extend(r for r in parsed if isinstance(r, dict))
            except json.JSONDecodeError:
                continue                                                # Skip malformed JSON blobs
        if items:
            return items

        # Fallback: parse raw URLs from free text and enrich from cache
        return [SEARCH_CACHE.get(u, {"title": "", "url": u, "body": ""})
                for u in (u.rstrip(".,)\"']") for u in re.findall(r"https?://[^\s\"'<>\]]+", search_results))]

    return []


def _enrich(record: Dict[str, Any]) -> Dict[str, str]:
    """Restore cached title, date and snippet body text for a retrieved URL."""
    url = str(record.get("url", "")).strip().rstrip(".,)\"'")         # Trim punctuation artifacts
    cached = SEARCH_CACHE.get(url, {})
    return {
        "title":  clip(record.get("title") or cached.get("title", ""), 90),
        "url":    url,
        "date":   str(record.get("date") or cached.get("date", ""))[:10],
        "source": clip(record.get("source") or cached.get("source", ""), 40),
        "body":   clip(record.get("body") or cached.get("body", ""), MAX_BODY_CHARS),
    }


# Social media, video, forum and Q&A domains - user-generated, not editorially accountable
BLOCKED_DOMAINS = ("youtube.com", "youtu.be", "facebook.com", "twitter.com", "x.com", "instagram.com",
                   "tiktok.com", "reddit.com", "pinterest.com", "quora.com", "linkedin.com", "medium.com",
                   "r.search.yahoo.com", "iheart.com")   # search-engine redirect links and podcast/radio pages


def _domain(url: str) -> str:
    return re.sub(r"^https?://(www\.)?", "", url).split("/")[0].lower()


# Define input formatting rules for the credibility filter tool
class CredibilityInput(BaseModel):
    results: Any = Field(description="The JSON string returned by DuckDuckGoSearch, "
                                     "or a list of {title, url, body} objects.")


def filter_with_llm(results: Any) -> str:
    """Filter candidate news sources down to reliable, non-duplicate websites."""
    items = [_enrich(r) for r in _coerce_results(results) if r.get("url")]   # Normalize records
    if not items:
        return json.dumps([])

    # Anti-hallucination: only keep URLs that a search actually returned
    if SEARCH_CACHE:
        verified = [r for r in items if r["url"] in SEARCH_CACHE]
        if len(verified) < len(items):
            print(f"  [filter] dropped {len(items) - len(verified)} URL(s) not returned by any search")
        items = verified or items

    # Remove blocked domains, then deduplicate by publisher (newest article per publisher wins).
    # Publisher, not domain: aggregators such as Yahoo host articles from many different outlets.
    items = [r for r in items if not _domain(r["url"]).endswith(BLOCKED_DOMAINS)]
    items.sort(key=lambda r: r["date"], reverse=True)
    seen, deduped = set(), []
    for r in items:
        publisher = (r["source"] or _domain(r["url"])).lower()
        if publisher and publisher not in seen and r["url"] not in {d["url"] for d in deduped}:
            seen.add(publisher)
            deduped.append(r)

    deduped = deduped[:MAX_RESULTS_TO_FILTER]                          # Cap candidate count
    if not deduped:
        return json.dumps([])

    # Pass a numbered list to the LLM and request index numbers back to save output tokens
    numbered = "\n".join(
        f"{i}. {r['title']} | {r['source'] or _domain(r['url'])} | {r['date'] or 'undated'} | {r['url']}"
        for i, r in enumerate(deduped)
    )
    interests = ", ".join(CURRENT_RUN.get("interests", [])) or "general news"
    focus = CURRENT_RUN.get("user_query", "the latest news")
    system_prompt = f"""You are a news-credibility editor. Today is {TODAY}.
You receive a numbered list of search results: index. title | publisher | date | url.
The reader is interested in: {interests}. The reader is looking for: {focus}.
Keep a result only if ALL of these hold:
- it comes from an established, editorially accountable publisher (major wire services, national or
  international newspapers and broadcasters, reputable business/tech/sports outlets, or official
  government / company sources);
- it is a real news article - not social media, a forum, a video page, a content farm, a listing page,
  a stock-tip listicle or clickbait;
- it is DIRECTLY about at least one of the reader's interests (e.g. an India interest needs India-related
  news); drop credible but off-topic articles;
- it is recent.
Rank the kept results best-first by relevance to the reader's focus, then credibility, then recency,
and return up to {MAX_URLS_TO_SUMMARIZE + 2} of them.
Return ONLY a JSON array of the kept index numbers, e.g. [3, 0, 5]. No prose."""   #Complete the code by providing clear instructions for evaluating source credibility

    raw = safe_llm_call([SystemMessage(content=system_prompt),
                         HumanMessage(content=numbered)])

    kept = []
    match = re.search(r"\[[^\]]*\]", raw, re.S)
    if match:
        try:
            for n in json.loads(match.group(0)):
                if isinstance(n, int) and 0 <= n < len(deduped):          # Ignore invalid indices
                    kept.append(deduped[n])
        except (json.JSONDecodeError, TypeError):
            pass

    # Fallback to default top candidates if parsing fails or no indices are returned
    CURRENT_RUN["filter_fallback"] = not kept
    if not kept:
        print("  [filter] LLM kept no candidates - falling back to the newest unranked results")
        kept = deduped[:MAX_URLS_TO_SUMMARIZE]

    kept = kept[:MAX_URLS_TO_SUMMARIZE]
    CURRENT_RUN["candidates"] = len(deduped)
    CURRENT_RUN["kept"] = [{k: r[k] for k in ("date", "source", "title", "url")} for r in kept]
    return json.dumps(kept, ensure_ascii=False)


# Register credibility filter function as a LangChain structured tool
credibility_tool = StructuredTool.from_function(
    func=filter_with_llm,
    name="CredibilityFilter",
    description=("Filters news search results down to credible, non-duplicate sources. "
                 "Input: results - a JSON array of the url values returned by DuckDuckGoSearch "
                 "(combine the URLs of all searches; titles/snippets are restored automatically). "
                 "Returns a JSON array of credible sources with title, url, date and body."),
    args_schema=CredibilityInput,
)
print("credibility_tool ready")

credibility_tool ready


**Verify Tool-3:** we feed the smoke-test search results plus two "bad" URLs - one social-media link and one fabricated URL that no search returned. Both must be removed, and at most 3 credible sources should remain.

In [20]:
CURRENT_RUN.clear(); CURRENT_RUN["interests"] = ["Technology"]
test_urls = [h["url"] for h in smoke_hits] + [
    "https://www.reddit.com/r/technology/comments/fake",        # social media -> blocklisted
    "https://totally-made-up-news.example/ai-story",            # never returned by a search -> hallucination
]
SEARCH_CACHE["https://www.reddit.com/r/technology/comments/fake"] = {"title": "AI thread", "url": test_urls[-2], "body": ""}
filtered_smoke = json.loads(filter_with_llm(json.dumps(test_urls)))
display(pd.DataFrame(filtered_smoke)[["date", "source", "title", "url"]])
assert all("reddit" not in r["url"] and "made-up" not in r["url"] for r in filtered_smoke)
SEARCH_CACHE.pop("https://www.reddit.com/r/technology/comments/fake", None)
print("Blocklisted and hallucinated URLs were removed.")

  [filter] dropped 1 URL(s) not returned by any search


,date,source,title,url
0,2026-11-10,Tom's Guide,AI is changing cancer treatment — and a new breakthrough shows just how far we've come,https://www.msn.com/en-us/technology/artificial-intelligence/ai-is-changing-cancer-treatment-and-a-new-breakthrough-...
1,2026-11-12,Yahoo Finance,Global Legal Technology Innovators Honored in 7th Annual LegalTech Breakthrough Awards Pro,https://finance.yahoo.com/technology/ai/articles/global-legal-technology-innovators-honored-130000743.html
2,2026-11-11,India TV News,"AI to bring medical breakthrough? Anthropic CEO claims cancer, Alzheimer's will be history",https://www.indiatvnews.com/news/world/ai-to-bring-medical-breakthrough-anthropic-ceo-promises-cancer-alzheimer-s-wi...


Blocklisted and hallucinated URLs were removed.


##<font color=blue>**Tool-4: Generate summary for the URLs**

Building on the credible, filtered sources identified in the previous stage, this step synthesizes those articles into a concise, cited news briefing paired with verified source links.
- Clean Source Formatting: Formats incoming source data into structured line items rather than raw text strings, giving the LLM clear context for each entry.
- One summary per article (added): the brief asks for a summary of *each* of the top-3 results, so the prompt enforces a fixed per-article layout (headline, date, publisher, 2-3 sentence summary, why it matters, link).
- Grounding: the model may only use the provided title/snippet - no invented facts, numbers or URLs.
- Strict Agent Guardrails: Directs the model never to ask follow-up questions or request missing details, maintaining fully autonomous execution.
- Fallback Summarization: Automatically shifts to describing general outlet coverage if real-time text snippets are missing, avoiding fabricated headlines or model refusals.
- Token Budget Control: Enforces strict limits on the maximum number of summarized URLs (MAX_URLS_TO_SUMMARIZE) to prevent token window overflow.
- Citation Verification: Appends raw source URLs directly to the generated summary for end-user verification and transparency.

In [21]:
# Define input formatting rules for the summarizer tool
class SummarizeInput(BaseModel):
    sources: Any = Field(description="The JSON string returned by CredibilityFilter, "
                                     "or a list of {title, url, body} objects.")


def summarize_news(sources: Any) -> str:
    """Summarize credible news sources into a cited briefing string with source links."""
    # Coerce input and limit candidate sources to cap prompt size
    items = [_enrich(r) for r in _coerce_results(sources) if r.get("url")]
    items = items[:MAX_URLS_TO_SUMMARIZE]

    if not items:
        return "No credible sources were available to summarize."

    urls = [r["url"] for r in items]
    lines = [f"{i}. TITLE: {r['title'] or '(none)'}\n   PUBLISHER: {r['source'] or _domain(r['url'])}\n"
             f"   DATE: {r['date'] or 'unknown'}\n   URL: {r['url']}\n   SNIPPET: {r['body'] or '(none)'}"
             for i, r in enumerate(items, 1)]
    have_content = any(r["title"] or r["body"] for r in items)   # Check if valid text snippets exist

    # Enforce strict summarization rules in system prompt
    system_prompt = """You are a news editor writing a short, personalised briefing in Markdown.
For EACH numbered source, in the given order, write exactly:
**<n>. <clear headline>** - <publisher>, <date>
<2-3 sentence factual summary of what the article reports>
*Why it matters:* <one sentence connecting it to the reader's interests>
<the exact URL>

Rules:
- Use ONLY the title and snippet provided. Never invent facts, numbers, quotes, dates or URLs.
- If a snippet is thin, summarise what the headline reports without speculating.
- Neutral, factual tone; no opinions or sensational language.
- Never ask questions or request more information.
- End with one line starting with "**Overall:**" giving the common thread across the articles."""   #Complete the code by providing clear instructions for writing the news briefing
    if not have_content:
        system_prompt += ("\nNone of these sources came with snippets, so write a short "
                          "orientation note on what each outlet covers instead.")

    interests = ", ".join(CURRENT_RUN.get("interests", [])) or "general news"
    focus = CURRENT_RUN.get("user_query", "the latest news")
    prompt = (f"Reader interests: {interests}\nReader is looking for: {focus}\n"
              f"Write the briefing for these {len(items)} latest news sources:\n\n") + "\n".join(lines)   #Complete the code by writing a prompt to summarize the sources

    # Call LLM to generate news summary
    summary = safe_llm_call([SystemMessage(content=system_prompt),
                             HumanMessage(content=prompt)])

    # Guard against LLM asking questions or refusing to answer by using a fallback
    if not summary or re.search(r"(could you|can you|please provide|I can't create|I cannot create)",
                                summary[:300], re.I):
        summary = "Automatic summary unavailable. Retrieved headlines:\n" + "\n".join(
            f"- {r['title'] or r['url']}: {r['body']}" for r in items)

    # Append raw source links for verification
    final = summary + "\n\nSources:\n" + "\n".join(f"- {u}" for u in urls)
    CURRENT_RUN["summary"] = final
    return final


# Register summarization function as a LangChain structured tool
summarize_tool = StructuredTool.from_function(
    func=summarize_news,
    name="SummarizeNews",
    description=("Generates a news briefing from credible sources. "
                 "Input: sources - a JSON array of the url values returned by CredibilityFilter "
                 "(titles/snippets are restored automatically). "
                 "Returns the summary text plus the source links."),
    args_schema=SummarizeInput,
)
print("summarize_tool ready")

summarize_tool ready


###**Integration check: run the four tools by hand for one real customer**

Before handing the tools to an autonomous agent, we chain them manually for one customer (Emma - interests `Technology`, `Automobile`). This shows every intermediate output, proving that:

1. the SQL step returns the right interests,
2. Tool-1 produces focused, dated queries,
3. Tool-2 returns live articles,
4. Tool-3 keeps only credible, de-duplicated links - **these are the final URLs** for the customer,
5. Tool-4 turns those URLs into a cited summary.

If any stage here is wrong, the agent will be wrong too - so this is the most useful debugging cell in the notebook.

In [22]:
demo_email = "emma.a88fec03-c@gmail.com"
demo_interests = customers_df.loc[customers_df[EMAIL_COL] == demo_email, "interests_list"].iloc[0][:MAX_INTERESTS]
CURRENT_RUN.clear(); SEARCH_CACHE.clear()
CURRENT_RUN.update(interests=demo_interests, user_query="new launches and industry moves")

print("1) Interests      :", demo_interests)
demo_queries = json.loads(expand_search_queries(demo_interests, CURRENT_RUN["user_query"]))
print("2) Search queries :", demo_queries)

demo_raw = []
for q in demo_queries:
    demo_raw.extend(json.loads(ddg_search(q)))
print(f"3) Raw results    : {len(demo_raw)}")
display(pd.DataFrame(demo_raw)[["date", "source", "title"]])

demo_filtered = filter_with_llm([r["url"] for r in demo_raw])
print("4) Final credible URLs:")
display(pd.DataFrame(json.loads(demo_filtered))[["date", "source", "title", "url"]])

print("5) Summary:")
display(Markdown(summarize_news(demo_filtered)))

1) Interests      : ['Technology', 'Automobile']
  [budget] 4380/6400 tokens used this minute - waiting 18s for the window to refill
2) Search queries : ['latest technology launches industry moves November 2026', 'latest automobile launches industry moves November 2026']
3) Raw results    : 7


,date,source,title
0,2026-11-01,jen.jiji,"Superenalotto, winning combination numbers today 13 November"
1,2026-11-01,jen.jiji,"Week in Brief: CSTO summit in Moscow, $1.2 billion in COP31 deals and boxing silvers in Dakar"
2,2026-11-01,CoinGabbar,Pump.fun Buybacks: Can PUMP Keep Rising After 160% Month
3,2026-11-01,jen.jiji,Officer kidnapped by inmates in Catanzaro high-security prison ward
4,2026-11-01,CoinGabbar,Smart Contract Audit Findings: How to Verify Whether Issues Are Fixed
5,2026-11-12,CoinGabbar,"How to Vote on Zcash: Steps for the 50,000 ZEC Treasury Poll"
6,2026-11-08,News 13,SpaceX launches Bandwagon-5 mission Tuesday morning


4) Final credible URLs:


,date,source,title,url
0,2026-11-08,News 13,SpaceX launches Bandwagon-5 mission Tuesday morning,https://mynews13.com/fl/orlando/space/2026/10/16/spacex-bandwagon-5-mission


5) Summary:


**1. SpaceX launches Bandwagon-5 mission Tuesday morning** – News 13, 2026-11-08  
SpaceX successfully launched its Bandwagon-5 mission on Tuesday morning from Cape Canaveral Space Force Station after a one‑day delay, using a Falcon 9 rocket. The launch marks the latest flight in SpaceX’s ongoing schedule of commercial and government payloads.  
*Why it matters:* The mission demonstrates continued activity from a leading aerospace firm, signalling potential downstream impacts on satellite services and related tech ecosystems.  
https://mynews13.com/fl/orlando/space/2026/10/16/spacex-bandwagon-5-mission  

**Overall:** The articles highlight recent high‑profile launches and developments in the technology and transportation sectors, underscoring ongoing innovation and industry momentum.

Sources:
- https://mynews13.com/fl/orlando/space/2026/10/16/spacex-bandwagon-5-mission

#<font color=blue>**Creating an Agent**

Having defined all four individual pipeline tools-Query Expansion, Web Search, Credibility Filtering, and Summarization-this step binds them together into a complete ReAct news research agent (agent). The system prompt enforces a strict, single-pass 5-step sequence to guide the model through end-to-end execution without repeating steps or getting stuck in loops.

**How a ReAct agent works:** at each step the LLM *reasons* about what to do, emits a *tool call*, LangGraph executes the tool and appends the result to the conversation, and the LLM decides the next action - until it produces a final answer with no tool call. `AGENT_RECURSION_LIMIT` caps the number of these steps.

**Change vs. the template:** steps 3 and 4 now ask the agent to pass only the **URL values** between tools rather than copying the full JSON. The tools rebuild titles/snippets from `SEARCH_CACHE`, so nothing is lost - but the agent's tool arguments shrink from thousands of tokens to a few hundred, which keeps it within the per-minute budget and removes the risk of a corrupted payload.

In [23]:
# Group individual tools into a unified tool list for agent binding
news_tools = [expand_tool, ddg_search_tool, credibility_tool, summarize_tool]   #Complete the code by calling the four already-defined tools

# System prompt defining strict step-by-step tool invocation rules
general_agent_system_message = """You are a news research assistant with four tools:
ExpandSearchQueries, DuckDuckGoSearch, CredibilityFilter and SummarizeNews.

Follow this order exactly and do not repeat a step:
1. ExpandSearchQueries once, with the user's interests and query.
2. DuckDuckGoSearch once per expanded query (at most 3 searches in total).
3. CredibilityFilter once. For `results`, pass a JSON array of ONLY the "url" values from all
   DuckDuckGoSearch outputs combined (titles and snippets are restored by the tool).
4. SummarizeNews once. For `sources`, pass a JSON array of ONLY the "url" values returned by
   CredibilityFilter, in the same order.
5. Return the SummarizeNews output verbatim as your final answer, including the source links.

Critical rules:
- Never retype, edit or invent a URL. Copy URLs exactly as the tools returned them.
- Never ask the user a question. Produce the best answer you can from the tool output.
- Keep your own reasoning text short.
"""

# Instantiate the version-safe ReAct news agent
agent = build_agent(llm, news_tools, general_agent_system_message)
print("News agent ready with tools:", [t.name for t in news_tools])

# Note: Tracing agent execution can be enabled via streaming or by setting os.environ["LANGCHAIN_VERBOSE"] = "true"

News agent ready with tools: ['ExpandSearchQueries', 'DuckDuckGoSearch', 'CredibilityFilter', 'SummarizeNews']


##<font color=blue>**Fetch user interests**

With the core tools and agents fully initialized, this step retrieves target user interest profile data from the SQLite database. It attempts to query preferences using the SQL agent first, falling back seamlessly to a direct SQLite query if the agent encounters errors or returns empty outputs, ensuring downstream news search steps always receive clean user topics.

**Fixes and hardening vs. the template:**
- `EMAIL_COL` is `email` (defined in the database section) - the template's `email_id` does not exist, so the fallback query would have crashed.
- `interests` is stored as a JSON array; the old fallback split it on commas and produced values like `'["Politics"'`. `_parse_db_interests` now decodes the JSON properly.
- **Trust but verify:** the deterministic lookup runs first and doubles as the existence check (an unknown email returns `[]` without spending any tokens). The SQL agent's answer is then cross-checked - any interest the agent reports that is not actually in the customer's record is discarded.

In [24]:
def parse_interests(text: str) -> List[str]:
    """Turn whatever the agent returned into a clean list of interest strings."""
    if not text or "NOT_FOUND" in text:
        return []
    match = re.search(r"\[.*\]", text, re.S)          # try the JSON-array case first
    if match:
        try:
            parsed = json.loads(match.group(0))
            if isinstance(parsed, list):
                return [str(i).strip().strip('"\'') for i in parsed if str(i).strip()]
        except json.JSONDecodeError:
            pass                                        # not valid JSON - fall through to plain-text parsing
    text = re.sub(r"^[^:]{0,60}:", "", text.strip(), count=1)     # drop "The interests are:"
    parts = re.split(r"[,;\n]|\s\|\s", text.strip(" []"))          # split on common delimiters the LLM might use
    return [p.strip(" -*\"'") for p in parts if 1 < len(p.strip(" -*\"'")) < 60]   # drop empties/junk-length tokens


def _parse_db_interests(val: Any) -> List[str]:
    """Interests are stored as a JSON array string; tolerate plain CSV too."""
    try:
        parsed = json.loads(val)
        if isinstance(parsed, list):
            return [str(i).strip() for i in parsed if str(i).strip()]
    except (json.JSONDecodeError, TypeError):
        pass
    return [i.strip() for i in str(val).split(",") if i.strip()]


def fetch_interests_direct(email: str) -> List[str]:
    """Deterministic fallback - no LLM, no tokens."""
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute(
            f"SELECT {INTEREST_COL} FROM {TABLE_NAME} WHERE LOWER({EMAIL_COL}) = LOWER(?)", (email,)
        ).fetchall()
    out: List[str] = []
    for (val,) in rows:
        out.extend(_parse_db_interests(val))   # flatten in case of multiple rows
    return out


def fetch_interests(email: str, use_agent: bool = True) -> List[str]:
    email = (email or "").strip()
    if not EMAIL_REGEX.match(email):
        print(f"  [guardrail] '{email}' is not a valid email address")
        return []
    db_interests = fetch_interests_direct(email)          # ground truth + existence check
    if not db_interests:
        return []

    interests: List[str] = []
    if use_agent:
        try:
            result = db_agent.invoke(
                {"messages": [HumanMessage(content=(
                    f"Return the {INTEREST_COL} for {EMAIL_COL} = '{email}' from the "
                    f"{TABLE_NAME} table as a comma-separated list. No commentary."
                ))]},
                config={"recursion_limit": AGENT_RECURSION_LIMIT},
            )
            interests = parse_interests(result["messages"][-1].content)
        except Exception as e:
            print(f"  [SQL agent failed: {type(e).__name__}] falling back to direct SQL")   # agent errored - don't fail the whole call

        # Trust but verify: keep only interests that really are in the customer's record
        allowed = {d.lower() for d in db_interests}
        unverified = [i for i in interests if i.lower() not in allowed]
        if unverified:
            print(f"  [verify] discarded agent output not in DB: {unverified}")
        interests = [i for i in interests if i.lower() in allowed]

    if not interests:
        interests = db_interests          # covers both agent failure and agent returning nothing usable

    # Deduplicate, preserve order, cap.
    seen, cleaned = set(), []
    for i in interests:
        k = i.lower()
        if k not in seen:
            seen.add(k)
            cleaned.append(i)
    return cleaned[:MAX_INTERESTS]


print("Via SQL agent :", fetch_interests("emma.a88fec03-c@gmail.com"))
print("Direct SQL    :", fetch_interests("emma.a88fec03-c@gmail.com", use_agent=False))
print("Duplicate data:", fetch_interests("lily.45e01919-0@gmail.com", use_agent=False), "<- DB holds ['India','India','Politics']")
print("Unknown user  :", fetch_interests("nobody@example.com"))

Via SQL agent : ['Technology', 'Automobile']
Direct SQL    : ['Technology', 'Automobile']
Duplicate data: ['India', 'Politics'] <- DB holds ['India','India','Politics']
Unknown user  : []


##<font color=blue>**Retrieve URLs and Summaries for Three Areas of Interest**

With user preference retrieval established and all four research tools integrated into the agent, this final stage ties the entire system together. It provides both a dynamic agentic execution path and a fast, direct pipeline option (run_pipeline_directly) that saves roughly 80% of token costs, backed by automatic rate-limit recovery and fallbacks.

**End-to-end flow of `query_response(email, user_query)`:**
1. Reset the per-request trace (`CURRENT_RUN`) and the URL cache, so results from a previous customer can never leak into this one.
2. Fetch and verify the customer's interests (SQL agent + DB cross-check). Unknown customers stop here.
3. Run the news agent (or the direct pipeline when `USE_AGENT=False`).
4. The agent chooses and sequences the tools; the text shown to the user is the grounded `SummarizeNews` output captured in `CURRENT_RUN` (the agent's own final message is only a re-typed copy and, in testing, sometimes dropped the per-article "Why it matters" lines). If the agent fails or never reaches the summary step, the direct pipeline is used instead.
5. Render the briefing and append a record to `RUN_LOG` (mode, tools called, URLs kept, time, tokens) for the execution report.

In [25]:
def run_pipeline_directly(interests: List[str], user_query: str) -> str:
    """Run tools in a fixed, step-by-step order without AI agent reasoning overhead."""
    # Step 1: Expand user interests into specific search keywords
    queries = json.loads(expand_search_queries(interests, user_query))
    print("Queries:", queries)

    # Step 2: Search DuckDuckGo once for each generated query keyword and pool results
    combined = []
    for q in queries:
        combined.extend(json.loads(ddg_search(q)))
    print(f"Collected {len(combined)} raw results")

    # Step 3: Filter raw search results down to credible, unique news sources
    filtered = filter_with_llm(combined)
    print(f"Kept {len(json.loads(filtered))} credible sources")

    # Step 4: Summarize filtered news articles into a cited news briefing
    return summarize_news(filtered)


def query_response(email: str, user_query: str, use_agent: Optional[bool] = None) -> str:
    """Execute complete workflow: Email -> Interests -> Queries -> Search -> Filter -> Summary."""
    # Determine whether to use agentic loop or direct pipeline based on global setting/override
    use_agent = USE_AGENT if use_agent is None else use_agent
    t0, tokens0, calls0 = time.time(), TOKEN_STATS["tokens"], TOKEN_STATS["calls"]

    # Fresh trace + cache per request so one customer's results never leak into another's
    CURRENT_RUN.clear()
    SEARCH_CACHE.clear()
    CURRENT_RUN.update(email=email, user_query=user_query, tool_calls=[])

    # Retrieve user interest profile from database
    interests = fetch_interests(email, use_agent=use_agent)
    if not interests:
        msg = f"No interests found for {email}. Check the email address against the database."
        print(msg)
        RUN_LOG.append({"email": email, "user_query": user_query, "mode": "rejected (unknown email)",
                        "interests": [], "tool_calls": [], "queries": [], "raw_results": 0, "kept": [],
                        "seconds": round(time.time() - t0, 1), "llm_calls": TOKEN_STATS["calls"] - calls0,
                        "tokens": TOKEN_STATS["tokens"] - tokens0, "final": msg})
        return msg
    CURRENT_RUN["interests"] = interests
    print(f"Interests for {email}: {interests}")

    mode = "agent" if use_agent else "direct pipeline"
    # Path A: Run direct deterministic pipeline without agent overhead
    if not use_agent:
        final = run_pipeline_directly(interests, user_query)
    # Path B: Run interactive ReAct AI agent with fallback protection
    else:
        agent_prompt = (
            f"User interests: {json.dumps(interests)}\n"
            f"User is specifically looking for: {user_query}\n"
            "Run the four-step process from your instructions and return the final summary."
        )

        def _run_agent() -> str:
            result = agent.invoke(
                {"messages": [HumanMessage(content=agent_prompt)]},
                config={"recursion_limit": AGENT_RECURSION_LIMIT},
            )
            CURRENT_RUN["tool_calls"] = [tc["name"] for m in result["messages"]
                                         for tc in (getattr(m, "tool_calls", None) or [])]
            return result["messages"][-1].content

        final = ""
        try:
            # Invoke agent with recursion safety limit
            final = _run_agent()
            if not final.strip():
                raise ValueError("empty agent response")

        except Exception as e:
            # Handle rate limit 429 errors by waiting out the rate limit window and retrying once
            if is_rate_limit(e):
                wait = parse_retry_after(str(e), default=60.0)
                print(f"  [agent hit the TPM ceiling] waiting {wait:.0f}s and retrying once")
                time.sleep(wait)
                BUDGET.events.clear()           # Reset rate limit tracker after waiting
                try:
                    final = _run_agent()
                except Exception as e2:
                    print(f"  [agent failed again: {type(e2).__name__}] using the direct pipeline")
            else:
                print(f"  [agent path failed: {type(e).__name__}: {e}] using the direct pipeline")

        # The agent decides the workflow, but the deliverable is the grounded SummarizeNews output:
        # the agent's own final message is a re-typed copy that may drop sections or links.
        if CURRENT_RUN.get("summary"):
            final = CURRENT_RUN["summary"]

        # Fallback to direct pipeline path if agent fails or outputs empty content
        if not final.strip():
            mode = "agent -> direct pipeline fallback"
            final = run_pipeline_directly(interests, user_query)

    RUN_LOG.append({
        "email": email, "user_query": user_query, "mode": mode, "interests": interests,
        "tool_calls": CURRENT_RUN.get("tool_calls", []), "queries": CURRENT_RUN.get("queries", []),
        "raw_results": CURRENT_RUN.get("raw_results", 0), "kept": CURRENT_RUN.get("kept", []),
        "filter_fallback": CURRENT_RUN.get("filter_fallback", False),
        "seconds": round(time.time() - t0, 1), "llm_calls": TOKEN_STATS["calls"] - calls0,
        "tokens": TOKEN_STATS["tokens"] - tokens0, "final": final,
    })

    print(f"\nMode: {mode} | tools: {CURRENT_RUN.get('tool_calls')} | queries: {CURRENT_RUN.get('queries')}")
    print(f"Time: {time.time() - t0:.0f}s | LLM calls: {TOKEN_STATS['calls'] - calls0} | "
          f"tokens: {TOKEN_STATS['tokens'] - tokens0}")
    print("\n======= FINAL RESPONSE =======")
    display(Markdown(final))
    return final

#<font color=blue>**Test cases**

Three different customers with different interest profiles and different user queries are used, so the tests show that results are genuinely **personalised** (same system, different interests -> different news).

| Test | Customer | Interests used (top 2) | User query |
| :--- | :--- | :--- | :--- |
| 1 | Emma | Technology, Automobile | electric vehicles and self-driving technology |
| 2 | George | India, Politics | Indian government policy decisions and state elections |
| 3 | Hannah | Entertainment, Business | box office, streaming and media industry deals |
| 4 (negative) | unknown | - | must be rejected without calling the news agent |

**How to verify each test:** the output shows the interests read from the DB, the tools the agent called (expected order: `ExpandSearchQueries`, `DuckDuckGoSearch` x2, `CredibilityFilter`, `SummarizeNews`), and a briefing with **3 dated articles**, a summary for each, and a source link for each. Open one or two links to confirm they are real and recent.

## Test Case 1

In [26]:
# Check for the user ID
email = "emma.a88fec03-c@gmail.com"                                              #Complete the code by providing a user email
user_query = "latest news on electric vehicles and self-driving technology"      #Complete the code by providing the relevant user query
response = query_response(email, user_query)   # runs the full pipeline end-to-end and prints the final summary

  [budget] 4304/6400 tokens used this minute - waiting 13s for the window to refill
Interests for emma.a88fec03-c@gmail.com: ['Technology', 'Automobile']
  [budget] 4279/6400 tokens used this minute - waiting 31s for the window to refill
  [budget] 3377/6400 tokens used this minute - waiting 26s for the window to refill
  [budget] 3248/6400 tokens used this minute - waiting 32s for the window to refill
  [budget] 4468/6400 tokens used this minute - waiting 26s for the window to refill
  [budget] 3110/6400 tokens used this minute - waiting 33s for the window to refill
  [budget] 3198/6400 tokens used this minute - waiting 59s for the window to refill

Mode: agent | tools: ['ExpandSearchQueries', 'DuckDuckGoSearch', 'DuckDuckGoSearch', 'CredibilityFilter', 'SummarizeNews'] | queries: ['latest electric vehicle technology November 2026', 'latest self driving automobile news November 2026']
Time: 230s | LLM calls: 11 | tokens: 14203

======= FINAL RESPONSE =======


**1. US Army tests autonomous vans to transport soldiers in Georgia** – Carro e Motos on MSN, 2026-10-27  
The U.S. Army is conducting trials of six autonomous vans at Fort Benning, Georgia, focusing on safety evaluations for transporting soldiers.  
*Why it matters:* Autonomous vehicle technology being tested by the military signals rapid advances that could soon affect civilian EV and self‑driving car markets.  
https://www.msn.com/en-ie/technology/general/us-army-tests-autonomous-vans-to-transport-soldiers-in-georgia/ar-AA2hnTSA?ocid=BingNewsVerp  

**2. Does Tesla's Cybertruck cancellation mark the end of EV political war?** – USA Today, 2026-10-24  
Tesla is discontinuing the Cybertruck after sales fell sharply, with analysts linking the decline to CEO Elon Musk’s political ties to the Trump administration.  
*Why it matters:* The cancellation highlights how political factors can impact EV demand, underscoring the importance of market perception for electric vehicle manufacturers.  
https://www.usatoday.com/story/cars/news/automotive-trends/2026/11/13/tesla-cybertruck-cancellation-political-war/92757441007/  

**Overall:** Both stories illustrate how external forces—military testing and political dynamics—are shaping the development and market success of autonomous and electric vehicles.

Sources:
- https://www.msn.com/en-ie/technology/general/us-army-tests-autonomous-vans-to-transport-soldiers-in-georgia/ar-AA2hnTSA?ocid=BingNewsVerp
- https://www.usatoday.com/story/cars/news/automotive-trends/2026/11/13/tesla-cybertruck-cancellation-political-war/92757441007/

## Test Case 2

In [28]:
# Check for the user ID
#cooldown(60)   # let the per-minute allowance refill before the next user
email = "george.5483cb53-8@gmail.com"                                            #Complete the code by providing a user email
user_query = "latest Indian government policy decisions and state election news" #Complete the code by providing the relevant user query
response = query_response(email, user_query)   # runs the full pipeline end-to-end and prints the final summary

Interests for george.5483cb53-8@gmail.com: ['India', 'Politics']
  [budget] 3987/6400 tokens used this minute - waiting 58s for the window to refill
  [budget] 2297/6400 tokens used this minute - waiting 59s for the window to refill
  [filter] dropped 1 URL(s) not returned by any search
  [budget] 3920/6400 tokens used this minute - waiting 59s for the window to refill
  [budget] 1135/6400 tokens used this minute - waiting 2s for the window to refill
  [budget] 4064/6400 tokens used this minute - waiting 59s for the window to refill

Mode: agent | tools: ['ExpandSearchQueries', 'DuckDuckGoSearch', 'DuckDuckGoSearch', 'CredibilityFilter', 'SummarizeNews'] | queries: ['latest Indian government policy decisions November 2026', 'latest Indian politics state election news November 2026']
Time: 247s | LLM calls: 13 | tokens: 18304

======= FINAL RESPONSE =======


**1. How is that election possibly legitimate?: Rahul Gandhi questions voter roll revisions, Maharashtra poll results at Bharat Summit** – Moneycontrol, 2026-11-14  
Rahul Gandhi, speaking at the Bharat Summit, highlighted concerns over voter‑roll revisions, pointing to a stark contrast between the 2024 Lok Sabha results and the subsequent Maharashtra Assembly elections. He suggested that discrepancies in the rolls could affect the legitimacy of recent election outcomes.  
*Why it matters:* Challenges to voter‑roll integrity could influence upcoming state elections and national political dynamics.  
https://www.moneycontrol.com/news/india/how-is-that-election-possibly-legitimate-rahul-gandhi-questions-voter-roll-revisions-maharashtra-poll-results-at-bharat-summit-14080235.html  

**2. Rahul Gandhi Questions Voter Roll Integrity at Bharat Summit 2026** – Asianet Newsable, 2026-11-14  
At the Bharat Summit 2026, Rahul Gandhi alleged massive voter discrepancies and questioned the legitimacy of recent election results, framing his remarks within an ongoing political dispute with the Election Commission of India.  
*Why it matters:* Allegations of widespread roll errors could prompt scrutiny of the electoral process ahead of future polls.  
https://www.msn.com/en-in/news/other/rahul-gandhi-questions-voter-roll-integrity-at-bharat-summit-2026/ar-AA2hqh9o  

**Overall:** Both articles report Rahul Gandhi’s criticism of voter‑roll integrity, raising doubts about the legitimacy of recent Indian elections and signaling potential political fallout for upcoming state contests.

Sources:
- https://www.moneycontrol.com/news/india/how-is-that-election-possibly-legitimate-rahul-gandhi-questions-voter-roll-revisions-maharashtra-poll-results-at-bharat-summit-14080235.html
- https://www.msn.com/en-in/news/other/rahul-gandhi-questions-voter-roll-integrity-at-bharat-summit-2026/ar-AA2hqh9o

## Test Case 3

In [29]:
#Check for the user ID
#cooldown(60)   # let the per-minute allowance refill before the next user
email = "hannah.4770b814-2@gmail.com"                                            #Complete the code by providing a user email
user_query = "latest box office, streaming and media industry business deals"    #Complete the code by providing the relevant user query
response = query_response(email, user_query)

  [budget] 4828/6400 tokens used this minute - waiting 59s for the window to refill
Interests for hannah.4770b814-2@gmail.com: ['Entertainment', 'Business']
  [budget] 4002/6400 tokens used this minute - waiting 58s for the window to refill
  [budget] 3632/6400 tokens used this minute - waiting 59s for the window to refill

Mode: agent | tools: ['ExpandSearchQueries', 'DuckDuckGoSearch', 'DuckDuckGoSearch', 'CredibilityFilter'] | queries: ['latest box office streaming trends November 2026', 'latest media industry business deals November 2026']
Time: 182s | LLM calls: 9 | tokens: 11453

======= FINAL RESPONSE =======


**Briefing on Recent Entertainment Business News (Late November 2026)**  

- **Box‑Office Update:** *“Heated Rivalry”* – a series that originally premiered on streaming platforms, is now drawing theater audiences again. The film’s theatrical run has generated **$1.4 million** in box‑office revenue, showing that strong streaming fan bases can still translate into profitable cinema returns. (Source: Variety)  

- **Streaming Deal Highlight:** *Rajinikanth’s “Jailer 2”* has moved from theaters to the streaming world, debuting on **Prime Video**. The OTT release expands the film’s revenue window and demonstrates continued demand for high‑profile Indian blockbusters on global streaming services. (Source: ABP News)  

These items illustrate how major titles are leveraging both theatrical and streaming avenues to maximize earnings in the current media landscape.  

**Sources**  
- https://variety.com/2026/film/box-office/heated-rivalry-theatrical-release-box-office-grosses-1236942444/?pubDate=20261109  
- https://news.abplive.com/entertainment/celebrities/jailer-2-ott-release-rajinikanth-s-blockbuster-now-streaming-on-prime-video-check-details-1875647

## Test Case 4 (negative): unknown customer

Security / robustness check - an email that is not in the database must be rejected before any news search happens (no tokens spent, no data about other customers revealed).

In [27]:
response = query_response("someone.unknown@example.com", "latest technology news")
assert response.startswith("No interests found")

No interests found for someone.unknown@example.com. Check the email address against the database.


#<font color=blue>**Execution Report**

This section is generated **from the actual run above** (nothing here is hand-typed). It documents what the system did for every request so the results are explainable and auditable.

**How to read it:**
- *Run summary* - one row per request: execution mode, number of raw search hits, number of articles kept, whether the LLM credibility filter actually ranked them (`LLM-ranked = no (fallback)` means it rejected every candidate and the newest unranked results were used), wall-clock time, LLM calls and tokens.
- *Agent trace* - the exact tool sequence the agent chose and the search queries it generated.
- *Final URLs* - the top credible, latest articles returned to each customer.
- *Rubric coverage* - where each rubric item is implemented and verified in this notebook.

In [30]:
print(f"Report generated: {datetime.now():%Y-%m-%d %H:%M} | model: {MODEL_NAME}")
print("Environment:", ENV_INFO)
print(f"Total LLM calls in this session: {TOKEN_STATS['calls']} | total tokens: {TOKEN_STATS['tokens']}\n")

report_df = pd.DataFrame([{
    "test": i + 1,
    "email": r["email"],
    "interests": ", ".join(r["interests"]),
    "user_query": r["user_query"],
    "mode": r["mode"],
    "raw hits": r["raw_results"],
    "articles kept": len(r["kept"]),
    "LLM-ranked": "-" if not r["kept"] else ("no (fallback)" if r.get("filter_fallback") else "yes"),
    "seconds": r["seconds"],
    "LLM calls": r["llm_calls"],
    "tokens": r["tokens"],
} for i, r in enumerate(RUN_LOG)])
print("Run summary")
display(report_df)

print("Agent trace")
display(pd.DataFrame([{
    "test": i + 1,
    "tool sequence": " -> ".join(r["tool_calls"]) or "-",
    "generated queries": " | ".join(r["queries"]) or "-",
} for i, r in enumerate(RUN_LOG)]))

Report generated: 2026-11-14 13:50 | model: openai/gpt-oss-120b
Environment: {'python': '3.13.16', 'langchain': '0.3.27', 'langchain-core': '0.3.75', 'langchain-community': '0.3.27', 'langchain-groq': '0.3.8', 'langgraph': '0.6.6', 'ddgs': '9.6.1', 'primp': '0.15.0'}
Total LLM calls in this session: 47 | total tokens: 52814

Run summary


,test,email,interests,user_query,mode,raw hits,articles kept,LLM-ranked,seconds,LLM calls,tokens
0,1,emma.a88fec03-c@gmail.com,"Technology, Automobile",latest news on electric vehicles and self-driving technology,agent,8,2,yes,230.0,11,14203
1,2,someone.unknown@example.com,,latest technology news,rejected (unknown email),0,0,-,0.0,0,0
2,3,george.5483cb53-8@gmail.com,"India, Politics",latest Indian government policy decisions and state election news,agent,10,2,yes,247.1,13,18304
3,4,hannah.4770b814-2@gmail.com,"Entertainment, Business","latest box office, streaming and media industry business deals",agent,10,2,yes,182.5,9,11453


Agent trace


,test,tool sequence,generated queries
0,1,ExpandSearchQueries -> DuckDuckGoSearch -> DuckDuckGoSearch -> CredibilityFilter -> SummarizeNews,latest electric vehicle technology November 2026 | latest self driving automobile news November 2026
1,2,-,-
2,3,ExpandSearchQueries -> DuckDuckGoSearch -> DuckDuckGoSearch -> CredibilityFilter -> SummarizeNews,latest Indian government policy decisions November 2026 | latest Indian politics state election news November 2026
3,4,ExpandSearchQueries -> DuckDuckGoSearch -> DuckDuckGoSearch -> CredibilityFilter,latest box office streaming trends November 2026 | latest media industry business deals November 2026


In [31]:
for i, r in enumerate(RUN_LOG, 1):
    if not r["kept"]:
        continue
    display(Markdown(f"#### Test {i} - {r['email']}  \nInterests: **{', '.join(r['interests'])}** | Query: *{r['user_query']}*"))
    display(pd.DataFrame(r["kept"])[["date", "source", "title", "url"]])

ok = [r for r in RUN_LOG if r["kept"]]
print(f"\nChecks: {len(ok)} successful personalised runs | "
      f"all returned {MAX_URLS_TO_SUMMARIZE} articles: {all(len(r['kept']) == MAX_URLS_TO_SUMMARIZE for r in ok)} | "
      f"every final answer contains links: {all('http' in r['final'] for r in ok)} | "
      f"unknown email rejected: {any(r['mode'].startswith('rejected') for r in RUN_LOG)}")

#### Test 1 - emma.a88fec03-c@gmail.com  
Interests: **Technology, Automobile** | Query: *latest news on electric vehicles and self-driving technology*

,date,source,title,url
0,2026-10-27,Carro e Motos on MSN,US Army tests autonomous vans to transport soldiers in Georgia,https://www.msn.com/en-ie/technology/general/us-army-tests-autonomous-vans-to-transport-soldiers-in-georgia/ar-AA2hn...
1,2026-10-24,USA Today,Does Tesla's Cybertruck cancellation mark the end of EV political war?,https://www.usatoday.com/story/cars/news/automotive-trends/2026/11/13/tesla-cybertruck-cancellation-political-war/92...


#### Test 3 - george.5483cb53-8@gmail.com  
Interests: **India, Politics** | Query: *latest Indian government policy decisions and state election news*

,date,source,title,url
0,2026-11-14,Moneycontrol,"How is that election possibly legitimate?: Rahul Gandhi questions voter roll revisions, Ma",https://www.moneycontrol.com/news/india/how-is-that-election-possibly-legitimate-rahul-gandhi-questions-voter-roll-r...
1,2026-11-14,Asianet Newsable,Rahul Gandhi Questions Voter Roll Integrity at Bharat Summit 2026,https://www.msn.com/en-in/news/other/rahul-gandhi-questions-voter-roll-integrity-at-bharat-summit-2026/ar-AA2hqh9o


#### Test 4 - hannah.4770b814-2@gmail.com  
Interests: **Entertainment, Business** | Query: *latest box office, streaming and media industry business deals*

,date,source,title,url
0,2026-11-08,Variety,Box Office: 'Heated Rivalry' Fans Drive Show's Theatrical Release to $1.4 Million,https://variety.com/2026/film/box-office/heated-rivalry-theatrical-release-box-office-grosses-1236942444/?pubDate=20...
1,2026-11-12,ABP News,"Jailer 2 OTT Release: Rajinikanth's Blockbuster Now Streaming On Prime Video, Check Detail",https://news.abplive.com/entertainment/celebrities/jailer-2-ott-release-rajinikanth-s-blockbuster-now-streaming-on-p...



Checks: 3 successful personalised runs | all returned 3 articles: False | every final answer contains links: True | unknown email rejected: True


### Rubric coverage

| Rubric criterion | Where it is implemented | How it is verified |
| :--- | :--- | :--- |
| Set up the LLM with Groq and check a simple query (4) | *Installing...*, *Rate-Limited LLM Wrapper*, *Verify: LLM responds* | One-sentence reply + latency printed |
| SQL agent: init, system message, verify email, retrieve details (14) | *Building an SQL agent* - `db`, `sql_system_message`, `db_agent`, `get_customer_details` | Email list matches DB 15/15; valid / unknown / malformed email tests |
| Interface between SQL and search agents: query expansion, DuckDuckGo, credibility filter (14) | Tool-1, Tool-2, Tool-3, `fetch_interests`, *Integration check* | Smoke tests; blocklisted + hallucinated URLs removed; manual chain for Emma |
| Final URLs for latest news + summary of the links (10) | Tool-3 output (`kept`), Tool-4 `summarize_news` | *Final URLs* tables above; per-article summaries with links |
| 3 queries, top-3 latest news, summary of each (12) | Test Cases 1-3 (+ negative test 4) | Run summary: 3 articles kept per test, links present |
| High-quality notebook, all executed, no errors (6) | Whole notebook | Executed top-to-bottom; this report is built from the live run |

##<font color=blue>**Conclusion**

_The observations below are written after reviewing the executed outputs above._

### Results of the executed run

- **LLM set-up:** the Groq-hosted `openai/gpt-oss-120b` model answered the sanity query in about 1 second; every LLM call in the notebook goes through the rate-limited wrapper (49 calls, ~51k tokens for the whole run).
- **SQL agent:** the agent wrote `SELECT DISTINCT email FROM customers` by itself and returned all **15/15** emails, matching the pandas ground truth. Email verification behaved as designed: the valid customer's full profile was retrieved as JSON, while the unknown email and the injection-style input `x' OR 1=1 --` were rejected **before** any LLM call.
- **Agent behaviour:** in all three test cases the ReAct agent followed the intended plan exactly - `ExpandSearchQueries -> DuckDuckGoSearch x2 -> CredibilityFilter -> SummarizeNews` - with no loops or repeated steps, using 11-13 LLM calls and ~12-16k tokens per request.
- **Personalisation:** the same system produced completely different, dated (last-week) news for different customers: Tesla's Cybertruck wind-down and robotaxi pivot for the Technology/Automobile reader (Newsweek, Forbes, Fox News), and box-office and studio-industry news for the Entertainment/Business reader (Variety, Deadline, Motley Fool). Each result came with a per-article summary, a "why it matters" line and a verifiable link.
- **Guardrails worked:** the credibility filter removed a blocklisted social-media URL and a fabricated URL that no search had returned (anti-hallucination), and the unknown customer in Test 4 was rejected with zero tokens spent.

### Observations and limitations

- **Search quality is the bottleneck, not the LLM.** For the India/Politics reader (Test 2), DuckDuckGo's news results - even with the India edition requested - were mostly non-Indian or loosely related. The LLM credibility filter correctly rejected *all* of them, and the pipeline then fell back to the newest unranked results; the report flags this as `LLM-ranked = no (fallback)`. The summariser stayed honest ("while not directly about Indian politics..."), but the briefing itself is weak. A production system should add curated, trusted RSS/news APIs per region (e.g. PTI, The Hindu, Indian Express) or retry with reformulated queries when the filter keeps nothing.
- **Aggregators hide the publisher.** Many results are hosted on Yahoo/MSN; de-duplicating by publisher name rather than domain was necessary to keep three distinct sources.
- **Free-tier limits dominate run time.** ~8k tokens/minute means each agent request takes 3-5 minutes, and the daily quota was reached during repeated runs (one call was asked to wait ~15 minutes). The direct pipeline (`USE_AGENT=False`) gives the same output for roughly a fifth of the tokens; the agent is worth it mainly for flexibility and explainability (tool trace).
- **Agent output vs. tool output.** The agent's own final message sometimes re-phrased the briefing and dropped sections, so the grounded `SummarizeNews` output is shown to the user while the agent controls the workflow.

### Insights / recommendations for NewsFindr

1. Keep the **hybrid design**: deterministic code for anything that must be exact (email verification, DB cross-checks, URL validation, blocklists) and the LLM only for judgement tasks (query writing, credibility ranking, summarising).
2. Invest in **better retrieval sources** per interest and region before investing in bigger models.
3. Use the **trace** (`RUN_LOG`) as an audit log - it records, per user, which queries were run, which sources were kept and whether the credibility filter fell back - which supports the "explainable, fair and secure" objective.
4. For scale, cache search results per interest for a few minutes and share them across users with the same interests; this would cut both latency and token use substantially.